# Занятие 9. Конспект: градиентный бустинг

Подробная версия семинара `seminar_09.ipynb`: выводы формул, функции
потерь регрессии и классификации, многоклассовый случай, AdaBoost,
регуляризация, устройство XGBoost (extreme gradient boosting), LightGBM
(light gradient boosting machine) и CatBoost (categorical boosting).

Содержание:

0. подготовка;
1. сумма базовых моделей и пошаговое построение;
2. градиентный спуск в пространстве функций;
3. функции потерь регрессии: MSE (mean squared error, среднеквадратичная ошибка),
   MAE (mean absolute error, средняя абсолютная ошибка), функция Хьюбера, квантильная;
4. бинарная классификация: логистическая потеря и шаг Ньютона;
5. многоклассовая классификация;
6. AdaBoost — бустинг с экспоненциальной потерей;
7. регуляризация: темп обучения, глубина, ранняя остановка, подвыборки;
8. смещение и разброс бустинга;
9. второй порядок и регуляризация листьев: XGBoost;
10. гистограммы, LightGBM и CatBoost;
11. важность признаков;
12. итоги.

## 0. Подготовка

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy.special import expit
from sklearn.datasets import load_wine, make_friedman1, make_hastie_10_2, make_moons
from sklearn.ensemble import (
    AdaBoostClassifier, BaggingRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
SEED = 42
LOWER_IS_BETTER = ("MSE", "ошибк", "время", "смещение", "разброс", "сумма")
NO_COLOR = ("число деревьев", "деревьев после", "дисперсия шума", "объектов", "сумма g", "сумма h", "вес", "ответ")


def show_table(frame, digits=3, axis=0):
    """Таблица с заливкой: лучшее значение ярче; для ошибок и времени лучше меньшее."""
    numeric = list(frame.select_dtypes("number").columns)
    plain = [c for c in numeric if any(key in str(c) for key in NO_COLOR)]
    lower = [c for c in numeric if c not in plain and any(key in str(c) for key in LOWER_IS_BETTER)]
    higher = [c for c in numeric if c not in plain and c not in lower]

    styled = frame.style.format(precision=digits, subset=numeric)
    if higher:
        styled = styled.background_gradient(cmap="Greens", subset=higher, axis=axis)
    if lower:
        styled = styled.background_gradient(cmap="Greens_r", subset=lower, axis=axis)
    return styled


SUBSCRIPT = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")


def tree_layout(structure):
    """Координаты узлов дерева: листья слева направо, внутренний узел — посередине между потомками."""
    positions = {}
    leaf_counter = [0]

    def place(node, depth):
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            positions[node] = (leaf_counter[0], -depth)
            leaf_counter[0] += 1
            return

        place(left_child, depth + 1)
        place(right_child, depth + 1)
        middle_x = (positions[left_child][0] + positions[right_child][0]) / 2
        positions[node] = (middle_x, -depth)

    place(0, 0)
    return positions


def draw_tree(ax, model, detailed=True):
    """Схема дерева: вопросы в узлах и значения листьев; для большого дерева — только узлы и ребра.

    У дерева регрессии в листе — среднее ответов, у классификатора — доля класса 1.
    """
    structure = model.tree_
    positions = tree_layout(structure)
    is_classifier = hasattr(model, "classes_")
    if structure.n_features == 1:
        feature_names = ["x"]
    else:
        feature_names = [f"x{j + 1}".translate(SUBSCRIPT) for j in range(structure.n_features)]

    for node, (x, y) in positions.items():
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]
        if left_child == -1:
            continue
        for child, answer in [(left_child, "да"), (right_child, "нет")]:
            child_x, child_y = positions[child]
            ax.plot([x, child_x], [y, child_y], color=GREY, lw=1.2 if detailed else 0.6, zorder=1)
            if detailed:
                ax.text((x + child_x) / 2, (y + child_y) / 2, answer, ha="center", va="center", fontsize=8,
                        color=GREY, bbox=dict(boxstyle="round,pad=0.1", facecolor="white", edgecolor="none"))

    for node, (x, y) in positions.items():
        is_leaf = structure.children_left[node] == -1

        if is_classifier:
            counts = structure.value[node, 0]
            leaf_value = counts[1] / counts.sum()            # доля класса 1 в листе
            leaf_text = f"p = {leaf_value:.2f}"
            is_positive = leaf_value > 0.5
        else:
            leaf_value = structure.value[node, 0, 0]
            leaf_text = f"{leaf_value:+.2f}"
            is_positive = leaf_value > 0

        if detailed and is_leaf:
            leaf_color = OCHRE if is_positive else BLUE
            ax.text(x, y, leaf_text, ha="center", va="center", fontsize=9, zorder=3,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor=leaf_color, alpha=0.25, edgecolor=leaf_color))
        elif detailed:
            feature_name = feature_names[structure.feature[node]]
            question = f"{feature_name} ≤ {structure.threshold[node]:.2f}?"
            ax.text(x, y, question, ha="center", va="center", fontsize=9, zorder=3,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=BLACK))
        else:
            ax.scatter(x, y, s=10 if is_leaf else 14, color=OCHRE if is_leaf else BLACK, zorder=2)

    n_leaves = model.get_n_leaves()
    lowest_level = min(y for _, y in positions.values())
    ax.set_xlim(-0.8, n_leaves - 0.2)
    ax.set_ylim(lowest_level - 0.6, 0.6)
    ax.axis("off")


def box(ax, x, y, text, color="white", edge=BLACK, size=11):
    """Прямоугольник с текстом для схем."""
    ax.text(x, y, text, ha="center", va="center", fontsize=size, zorder=3,
            bbox=dict(boxstyle="round,pad=0.45", facecolor=color, edgecolor=edge))


def arrow(ax, start, end, color=GREY, bend=0.0):
    """Стрелка от start к end; bend — изгиб дуги."""
    ax.annotate("", xy=end, xytext=start, zorder=2,
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5, connectionstyle=f"arc3,rad={bend}"))

## 1. Сумма базовых моделей и пошаговое построение

Модель бустинга — взвешенная сумма $T$ базовых моделей $b(x; \theta)$ из одного класса,
например деревьев с параметрами $\theta$ (вопросы узлов и значения листьев):

$$a_T(x) = a_0 + \sum_{t=1}^{T} \rho_t\, b(x; \theta_t).$$

Подобрать все $\rho_t$ и $\theta_t$ одновременно, минимизируя $\sum_{i=1}^n L(y_i, a_T(x_i))$, слишком сложно.
**Пошаговое построение** (forward stagewise additive modeling) добавляет модели
по одной и не меняет уже добавленные. В англоязычной литературе такую сумму
называют аддитивной моделью по базовым моделям; в разделе 7 и в глоссарии
аддитивная модель — сумма функций отдельных признаков.

$$(\rho_t, \theta_t) = \arg\min_{\rho,\, \theta} \sum_{i=1}^{n} L\big(y_i,\ a_{t-1}(x_i) + \rho\, b(x_i; \theta)\big), \qquad a_t(x) = a_{t-1}(x) + \rho_t\, b(x; \theta_t).$$

**Квадратичная потеря.** Для $L(y, a) = \frac{1}{2}(y - a)^2$ и $\rho = 1$ сумма на шаге $t$:

$$\sum_{i=1}^{n} \tfrac{1}{2}\big(y_i - a_{t-1}(x_i) - b(x_i; \theta)\big)^2 = \sum_{i=1}^{n} \tfrac{1}{2}\big(r_{ti} - b(x_i; \theta)\big)^2, \qquad r_{ti} = y_i - a_{t-1}(x_i).$$

Шаг пошагового построения для квадратичной потери — обучить модель
на **остатках** $r_{ti}$ по MSE. Дерево регрессии обучают именно так (занятие 8), но жадно:
разбиения выбирают по одному, и наилучшее дерево алгоритм CART (classification and regression
trees) не находит. Поэтому шаг выполняют приближенно; точны только значения листьев
при найденных разбиениях — средние остатков. Синус с шумом из занятия 8: 80 точек обучения,
2000 точек тестового сета, $\varepsilon \sim \mathcal{N}(0, 0.25^2)$.

In [ ]:
rng_sine = np.random.default_rng(0)

x_sine = np.sort(rng_sine.uniform(0, 2 * np.pi, 80))[:, None]        # (80, 1): обучающая выборка
noise_sine = rng_sine.normal(0, 0.25, 80)
y_sine = np.sin(x_sine[:, 0]) + noise_sine

x_sine_test = rng_sine.uniform(0, 2 * np.pi, 2000)[:, None]          # (2000, 1): тестовый сет
noise_sine_test = rng_sine.normal(0, 0.25, 2000)
y_sine_test = np.sin(x_sine_test[:, 0]) + noise_sine_test

x_grid = np.linspace(0, 2 * np.pi, 400)[:, None]                      # (400, 1): точки для рисунков

print("обучение:", x_sine.shape, " тест:", x_sine_test.shape)

**Пень** (decision stump) — дерево глубины 1: один вопрос и два листа; **глубокое дерево** —
дерево без ограничения глубины, у которого в листьях остаются по одному-два объекта.
Устройство трех деревьев и их предсказания на синусе:

In [ ]:
depth_cases = [(1, "пень"), (3, "дерево глубины 3"), (None, "глубокое дерево: глубина не ограничена")]

fig, axes = plt.subplots(2, 3, figsize=(19, 9.5), gridspec_kw={"height_ratios": [1, 1.15]})

for column, (max_depth, name) in enumerate(depth_cases):
    model = DecisionTreeRegressor(max_depth=max_depth, random_state=SEED)
    model.fit(x_sine, y_sine)

    n_leaves = model.get_n_leaves()
    depth = model.get_depth()
    test_prediction = model.predict(x_sine_test)
    test_mse = np.mean((test_prediction - y_sine_test) ** 2)
    is_small = n_leaves <= 8

    ax_tree = axes[0, column]
    draw_tree(ax_tree, model, detailed=is_small)
    ax_tree.set_title(f"{name}\nглубина {depth}, вопросов {n_leaves - 1}, листьев {n_leaves}", fontsize=13)

    ax_fit = axes[1, column]
    grid_prediction = model.predict(x_grid)
    ax_fit.scatter(x_sine[:, 0], y_sine, s=14, color=GREY, label="обучающая выборка")
    ax_fit.plot(x_grid[:, 0], np.sin(x_grid[:, 0]), color=OCHRE, lw=1.5, ls="--", label="sin x")
    ax_fit.plot(x_grid[:, 0], grid_prediction, color=BLACK, lw=2, label="предсказание дерева")
    if is_small:
        thresholds = model.tree_.threshold[model.tree_.children_left != -1]
        for threshold in thresholds:
            ax_fit.axvline(threshold, color=BLUE, ls=":", lw=1)
    ax_fit.set_title(f"MSE на тестовом сете {test_mse:.3f}", fontsize=12)
    ax_fit.set_xlabel("x")
    if column == 0:
        ax_fit.set_ylabel("y")
        ax_fit.legend(loc="lower left", fontsize=9)

plt.tight_layout()
plt.show()

MSE на тестовом сете: 0.176 у пня (2 листа), 0.087 у дерева глубины 3 (8 листьев) и 0.125
у глубокого дерева (80 листьев, глубина 11): пень недообучен, глубокое дерево повторяет шум.

Бэггинг и бустинг устроены по-разному: в бэггинге деревья обучают независимо на бутстрэп-выборках
и усредняют, в бустинге — по очереди, каждое на остатках суммы предыдущих, и складывают.

In [ ]:
fig, (ax_bagging, ax_boosting) = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={"height_ratios": [1, 1.25]})

# --- бэггинг: деревья независимы, ответ — среднее ---
ax_bagging.set_xlim(0, 10)
ax_bagging.set_ylim(0, 4)
ax_bagging.axis("off")
ax_bagging.set_title("бэггинг: деревья обучают независимо, ответ — среднее", fontsize=14, loc="left")

box(ax_bagging, 0.9, 2, "обучающая\nвыборка", color="#EEF2F6")
rows = [(3.3, "1"), (2.0, "2"), (0.7, "B")]
for y, number in rows:
    box(ax_bagging, 3.2, y, f"бутстрэп-выборка {number}")
    box(ax_bagging, 5.9, y, f"глубокое дерево $b_{number}$", color=BLUE + "33", edge=BLUE)
    arrow(ax_bagging, (1.55, 2), (2.1, y))
    arrow(ax_bagging, (4.35, y), (4.9, y))
    arrow(ax_bagging, (6.9, y), (7.75, 2))
box(ax_bagging, 8.8, 2, "$a(x) = \\frac{1}{B}\\,(b_1 + b_2 + \\dots + b_B)$", color=OCHRE + "33", edge=OCHRE, size=12)

# --- бустинг: деревья по очереди, каждое — на остатках суммы предыдущих ---
ax_boosting.set_xlim(0, 10)
ax_boosting.set_ylim(0, 5)
ax_boosting.axis("off")
ax_boosting.set_title("бустинг: деревья обучают по очереди, каждое — на ошибках суммы предыдущих, ответ — сумма",
                      fontsize=14, loc="left")

box(ax_boosting, 0.8, 1.0, "$a_0 = \\bar{y}$", color="#EEF2F6")
step_x = [2.6, 5.0, 7.4]
for t, x in enumerate(step_x, start=1):
    box(ax_boosting, x, 4.1, f"остатки\n$r_{t} = y - a_{t - 1}$", color=OCHRE + "33", edge=OCHRE)
    box(ax_boosting, x, 2.55, f"пень $b_{t}$,\nобученный на $r_{t}$", color=BLUE + "33", edge=BLUE)
    box(ax_boosting, x, 1.0, f"$a_{t} = a_{t - 1} + b_{t}$", color="#EEF2F6")
    arrow(ax_boosting, (x, 3.55), (x, 3.05))
    arrow(ax_boosting, (x, 2.05), (x, 1.4))
    previous_x = 0.8 if t == 1 else step_x[t - 2]
    arrow(ax_boosting, (previous_x + 0.45, 1.2), (x - 0.55, 3.75))             # от a_{t-1} к остаткам r_t
    arrow(ax_boosting, (previous_x + 0.55, 1.0), (x - 0.75, 1.0))              # a_{t-1} входит в a_t
box(ax_boosting, 9.4, 1.0, "…  $a_T = a_0 + b_1 + \\dots + b_T$", color=OCHRE + "33", edge=OCHRE, size=12)
arrow(ax_boosting, (7.95, 1.0), (8.35, 1.0))

plt.tight_layout()
plt.show()

Сто шагов с пнями и $\rho = 1$; на каждом шаге сохраняют пень, остатки и предсказания:

In [ ]:
n_steps = 100

initial_value = y_sine.mean()
current_train = np.full(len(y_sine), initial_value)          # a_{t-1}(x_i) на обучающей выборке
current_test = np.full(len(y_sine_test), initial_value)      # a_{t-1}(x) на тестовом сете
current_grid = np.full(len(x_grid), initial_value)           # a_{t-1}(x) в точках рисунка

stumps = []                      # b_1, ..., b_T
residuals_by_step = []           # r_t: остатки, на которых обучен пень b_t
models_on_train = [current_train]   # a_0, a_1, ..., a_T на обучающей выборке
models_on_grid = [current_grid]     # a_0, a_1, ..., a_T в точках рисунка
train_mse = [np.mean((y_sine - current_train) ** 2)]
test_mse = [np.mean((y_sine_test - current_test) ** 2)]

for step in range(n_steps):
    residuals = y_sine - current_train

    stump = DecisionTreeRegressor(max_depth=1, random_state=SEED)
    stump.fit(x_sine, residuals)

    current_train = current_train + stump.predict(x_sine)
    current_test = current_test + stump.predict(x_sine_test)
    current_grid = current_grid + stump.predict(x_grid)

    stumps.append(stump)
    residuals_by_step.append(residuals)
    models_on_train.append(current_train)
    models_on_grid.append(current_grid)
    train_mse.append(np.mean((y_sine - current_train) ** 2))
    test_mse.append(np.mean((y_sine_test - current_test) ** 2))

best_test_step = int(np.argmin(test_mse))
for T in [0, 1, 3, 10, 100]:
    print(f"T = {T:3d}: MSE на обучении {train_mse[T]:.4f}, на тесте {test_mse[T]:.4f}")
print(f"наименьшая MSE на тесте {test_mse[best_test_step]:.4f} при T = {best_test_step}")

Первые четыре шага, по столбцу на шаг: модель до шага и ее остатки (оранжевые отрезки),
остатки и пень, обученный на них, модель после шага.

In [ ]:
def stump_rule(stump):
    """Текст правила пня: вопрос и значения двух листьев."""
    structure = stump.tree_
    threshold = structure.threshold[0]
    left_value = structure.value[structure.children_left[0], 0, 0]
    right_value = structure.value[structure.children_right[0], 0, 0]
    rule = f"x ≤ {threshold:.2f}?\nда: {left_value:+.2f}\nнет: {right_value:+.2f}"
    return rule, threshold


shown_steps = [1, 2, 3, 4]
fig, axes = plt.subplots(3, len(shown_steps), figsize=(20, 12.5), sharex=True)

for column, step in enumerate(shown_steps):
    previous_train = models_on_train[step - 1]
    previous_grid = models_on_grid[step - 1]
    new_grid = models_on_grid[step]
    residuals = residuals_by_step[step - 1]
    stump = stumps[step - 1]
    rule, threshold = stump_rule(stump)

    # 1: модель до шага; вертикальные отрезки — остатки y_i − a_{t−1}(x_i)
    ax = axes[0, column]
    ax.vlines(x_sine[:, 0], previous_train, y_sine, color=OCHRE, lw=1, alpha=0.8)
    ax.scatter(x_sine[:, 0], y_sine, s=12, color=GREY, zorder=3)
    ax.plot(x_grid[:, 0], previous_grid, color=BLUE, lw=2, label=f"$a_{step - 1}$")
    ax.set_title(f"шаг {step}: модель $a_{step - 1}$ и ее остатки", fontsize=12)
    ax.legend(loc="lower left")

    # 2: остатки как точки и пень, обученный на них
    ax = axes[1, column]
    ax.axhline(0, color=GREY, lw=0.8)
    ax.scatter(x_sine[:, 0], residuals, s=12, color=OCHRE, zorder=3, label=f"остатки $r_{step}$")
    ax.plot(x_grid[:, 0], stump.predict(x_grid), color=BLUE, lw=2.5, label=f"пень $b_{step}$")
    ax.axvline(threshold, color=BLUE, ls=":", lw=1)
    ax.text(0.97, 0.96, rule, transform=ax.transAxes, ha="right", va="top", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=BLUE))
    ax.set_title(f"пень $b_{step}$, обученный на остатках", fontsize=12)
    ax.legend(loc="lower left")

    # 3: новая модель — старая плюс пень
    ax = axes[2, column]
    ax.scatter(x_sine[:, 0], y_sine, s=12, color=GREY, zorder=3)
    ax.plot(x_grid[:, 0], previous_grid, color=BLUE, lw=1.5, ls="--", label=f"$a_{step - 1}$")
    ax.plot(x_grid[:, 0], new_grid, color=BLACK, lw=2, label=f"$a_{step} = a_{step - 1} + b_{step}$")
    ax.set_title(f"MSE на обучении: {train_mse[step - 1]:.3f} → {train_mse[step]:.3f}", fontsize=12)
    ax.set_xlabel("x")
    ax.legend(loc="lower left")

axes[0, 0].set_ylabel("y")
axes[1, 0].set_ylabel("остаток")
axes[2, 0].set_ylabel("y")
plt.tight_layout()
plt.show()

Первый пень с вопросом «$x \le 3.50$?» снижает MSE на обучающей выборке с 0.504 до 0.126; второй
поднимает правый край $x > 5.78$ на 0.36, третий — участок $x \le 2.45$ на 0.17, четвертый опускает
левый край $x \le 0.45$ на 0.52.

Первые шесть пней по отдельности, их сумма после 1, 6 и 100 пней и MSE от числа пней:

In [ ]:
fig, (ax_parts, ax_sum, ax_error) = plt.subplots(1, 3, figsize=(20, 5))

part_colors = plt.cm.viridis(np.linspace(0, 0.85, 6))
for number, (stump, color) in enumerate(zip(stumps[:6], part_colors), start=1):
    ax_parts.plot(x_grid[:, 0], stump.predict(x_grid), color=color, lw=2, label=f"$b_{{{number}}}$")
ax_parts.axhline(0, color=GREY, lw=0.8)
ax_parts.set_title("первые 6 пней по отдельности", fontsize=12)
ax_parts.set_xlabel("x")
ax_parts.legend(ncol=2, fontsize=9)

ax_sum.scatter(x_sine[:, 0], y_sine, s=12, color=GREY, label="обучающая выборка")
ax_sum.plot(x_grid[:, 0], np.sin(x_grid[:, 0]), color=OCHRE, lw=2, ls="--", label="sin x")
for T, color in zip([1, 6, 100], [GREY, BLUE, BLACK]):
    ax_sum.plot(x_grid[:, 0], models_on_grid[T], color=color, lw=1.8, label=f"$a_{{{T}}}$")
ax_sum.set_title("сумма пней", fontsize=12)
ax_sum.set_xlabel("x")
ax_sum.legend(loc="lower left", fontsize=9)

steps = np.arange(n_steps + 1)
ax_error.plot(steps, train_mse, color=BLUE, lw=2, label="обучение")
ax_error.plot(steps, test_mse, color=OCHRE, lw=2, label="тест")
ax_error.axhline(0.25 ** 2, color=GREY, ls="--", label=r"дисперсия шума $\sigma_\varepsilon^2$")
ax_error.scatter(best_test_step, test_mse[best_test_step], color=BLACK, zorder=3, label=f"минимум на тесте: T = {best_test_step}")
ax_error.set_yscale("log")
ax_error.set_yticks([0.02, 0.05, 0.1, 0.2, 0.5])
ax_error.set_yticklabels(["0.02", "0.05", "0.1", "0.2", "0.5"])
ax_error.minorticks_off()
ax_error.set_xlabel("число пней T")
ax_error.set_ylabel("MSE")
ax_error.set_title("ошибка от числа пней", fontsize=12)
ax_error.legend()

plt.tight_layout()
plt.show()

**Почему ошибка на обучении не растет.** Значение листа дерева — среднее $\bar{r}$ остатков
его объектов. Для любого набора чисел

$$\sum_i r_i^2 = \sum_i (r_i - \bar{r} + \bar{r})^2 = \sum_i (r_i - \bar{r})^2 + 2\bar{r}\sum_i (r_i - \bar{r}) + k\,\bar{r}^2 = \sum_i (r_i - \bar{r})^2 + k\,\bar{r}^2,$$

где $k$ — число объектов листа, а сумма отклонений от среднего $\sum_i (r_i - \bar{r})$ равна нулю. Поэтому после шага
сумма квадратов остатков в каждом листе уменьшается на $k\,\bar{r}^2 \ge 0$. MSE на обучении
к 100 пням падает до 0.022, ниже дисперсии шума 0.0625, а на тестовом сете наименьшая
при $T = 56$ (0.086) и к $T = 100$ растет до 0.090: модель начинает повторять шум обучающей выборки.

Тот же алгоритм в sklearn — `GradientBoostingRegressor` с `learning_rate=1.0`:

In [ ]:
library_model = GradientBoostingRegressor(n_estimators=n_steps, learning_rate=1.0, max_depth=1, random_state=SEED)
library_model.fit(x_sine, y_sine)

library_train = library_model.predict(x_sine)
library_grid = library_model.predict(x_grid)

difference_train = np.max(np.abs(library_train - current_train))
difference_grid = np.max(np.abs(library_grid - models_on_grid[-1]))

print(f"начальное приближение sklearn {library_model.init_.constant_[0, 0]:.6f}, среднее y {initial_value:.6f}")
print(f"наибольшее расхождение с ручным циклом: обучение {difference_train:.1e}, точки рисунка {difference_grid:.1e}")

## 2. Градиентный спуск в пространстве функций

**Градиентный спуск по параметрам.** Для модели с параметрами $w$ и функционала
$Q(w) = \sum_i L(y_i, a(x_i; w))$ шаг спуска — $w \leftarrow w - \eta \nabla_w Q(w)$ (занятия 2 и 3).

**Спуск по предсказаниям.** Значения модели на обучающей выборке образуют вектор
$\mathbf{a} = (a_1, \dots, a_n)$, $a_i = a(x_i)$. Функционал $Q(\mathbf{a}) = \sum_i L(y_i, a_i)$ зависит от каждой
координаты через свое слагаемое, поэтому

$$\frac{\partial Q}{\partial a_i} = \frac{\partial L(y_i, a_i)}{\partial a_i}, \qquad -\nabla_{\mathbf{a}} Q = \Big(-\frac{\partial L(y_1, a_1)}{\partial a_1},\ \dots,\ -\frac{\partial L(y_n, a_n)}{\partial a_n}\Big) = (r_1, \dots, r_n).$$

Шаг спуска $a_i \leftarrow a_i + \eta\, r_i$ определен только в точках обучающей выборки. Чтобы
получить функцию, определенную всюду, вектор антиградиента **приближают**
базовой моделью: ищут $b$ из класса деревьев, значения которого на обучающей
выборке ближе всего к $(r_1, \dots, r_n)$ в смысле суммы квадратов,

$$b_t = \arg\min_{b} \sum_{i=1}^{n} \big(r_{ti} - b(x_i)\big)^2.$$

Это обычное обучение дерева регрессии на парах $(x_i, r_{ti})$. Компоненты $r_{ti}$ называют
**псевдоостатками**: для квадратичной потери они равны остаткам, для других — нет.

**Длина шага.** Направление $b_t$ задает дерево, длину — одномерная оптимизация
исходной потери (line search):

$$\rho_t = \arg\min_{\rho} \sum_{i=1}^{n} L\big(y_i,\ a_{t-1}(x_i) + \rho\, b_t(x_i)\big).$$

**Квадратичная потеря.** Псевдоостаток — остаток, $r_{ti} = y_i - a_{t-1}(x_i)$, а дерево
регрессии, обученное по MSE, предсказывает в каждом листе среднее остатков. Множитель $\rho_t$
для $L = \frac{1}{2}(y - a)^2$ и $b_i = b(x_i; \theta_t)$:

$$\rho_t = \arg\min_{\rho} \sum_{i=1}^{n} \tfrac{1}{2}\big(y_i - a_{t-1}(x_i) - \rho\, b_i\big)^2 = \arg\min_{\rho} \sum_{i=1}^{n} \tfrac{1}{2}(r_{ti} - \rho\, b_i)^2.$$

Производная по $\rho$ равна $-\sum_i b_i (r_{ti} - \rho\, b_i)$ и обращается в ноль при

$$\rho_t = \frac{\sum_i r_{ti}\, b_i}{\sum_i b_i^2}.$$

Пусть $c_j$ — значение листа $R_j$, среднее остатков в нем: $\sum_{x_i \in R_j} r_{ti} = |R_j|\, c_j$.
У объектов листа $b_i = c_j$, поэтому числитель и знаменатель равны:

$$\sum_i r_{ti}\, b_i = \sum_{j} c_j \sum_{x_i \in R_j} r_{ti} = \sum_{j} |R_j|\, c_j^2 = \sum_i b_i^2.$$

Следовательно, $\rho_t = 1$, и алгоритм совпадает с бустингом на остатках из раздела 1.

Дерево с $J$ листьями — сумма $J$ индикаторов областей, $b_t(x) = \sum_{j} c_{tj}\,[x \in R_{tj}]$ (занятие 8),
поэтому вместо одного $\rho_t$ на все дерево Фридман предложил подбирать свой сдвиг
для каждого листа (TreeBoost):

$$\gamma_{tj} = \arg\min_{\gamma} \sum_{x_i \in R_{tj}} L\big(y_i,\ a_{t-1}(x_i) + \gamma\big).$$

Каждый объект попадает ровно в один лист, поэтому сумма потерь распадается на суммы по листьям:

$$\sum_{i=1}^{n} L\Big(y_i,\ a_{t-1}(x_i) + \sum_{j=1}^{J} \gamma_{tj}\, [x_i \in R_{tj}]\Big) = \sum_{j=1}^{J} \sum_{x_i \in R_{tj}} L\big(y_i,\ a_{t-1}(x_i) + \gamma_{tj}\big).$$

Слагаемые разных листьев не зависят друг от друга, и задача распадается на $J$ задач
с одной переменной. Итоговый алгоритм **градиентного бустинга** (Friedman, 2001):

1. $a_0 = \arg\min_c \sum_i L(y_i, c)$;
2. для $t = 1, \dots, T$:
   - $r_{ti} = -\partial L(y_i, a)/\partial a$ при $a = a_{t-1}(x_i)$;
   - дерево регрессии на парах $(x_i, r_{ti})$ по MSE задает области $R_{t1}, \dots, R_{tJ}$;
   - $\gamma_{tj} = \arg\min_\gamma \sum_{x_i \in R_{tj}} L(y_i, a_{t-1}(x_i) + \gamma)$;
   - $a_t(x) = a_{t-1}(x) + \eta \sum_j \gamma_{tj}\,[x \in R_{tj}]$.

Множитель $\eta \in (0, 1]$ — темп обучения (раздел 7). Схема одного шага:

In [ ]:
LIGHT = "#EEF2F6"


def flow(ax, texts, colors, half_widths, title, footnote, gap=0.55):
    """Шаги одного круга алгоритма слева направо и стрелка обратно к первому шагу."""
    centers = []
    cursor = 0.1
    for half_width in half_widths:
        centers.append(cursor + half_width)
        cursor = cursor + 2 * half_width + gap

    ax.set_xlim(0, cursor - gap + 0.1)
    ax.set_ylim(0, 3)
    ax.axis("off")
    ax.set_title(title, fontsize=14, loc="left")

    for x, text, (face, edge) in zip(centers, texts, colors):
        box(ax, x, 1.9, text, color=face, edge=edge)
    for left in range(len(centers) - 1):
        start = centers[left] + half_widths[left] + 0.05
        end = centers[left + 1] - half_widths[left + 1] - 0.05
        arrow(ax, (start, 1.9), (end, 1.9))
    arrow(ax, (centers[-1], 1.2), (centers[0], 1.2), bend=-0.06)
    ax.text((cursor - gap) / 2, 0.15, footnote, ha="center", fontsize=11, color=GREY)


# --- раздел 1: смещение и разброс -----------------------------------------------


def plot_gb_scheme():
    """Один шаг градиентного бустинга и таблица псевдоостатков и значений листьев."""
    fig, (ax_flow, ax_table) = plt.subplots(2, 1, figsize=(18, 7.5), gridspec_kw={"height_ratios": [1.15, 1]})

    texts = [
        "текущая модель\n$a_{t-1}(x)$",
        "псевдоостатки\n$r_i = -\\partial L(y_i, a) / \\partial a$\nв точке $a = a_{t-1}(x_i)$",
        "дерево по MSE\nна парах $(x_i, r_i)$:\nобласти $R_1, \\dots, R_J$",
        "значение каждого листа\nпо исходной потере $L$:\n"
        "$\\gamma_j = \\mathrm{argmin}_\\gamma \\sum_{x_i \\in R_j} L(y_i, a_{t-1}(x_i) + \\gamma)$",
        "новая модель\n$a_t(x) = a_{t-1}(x) + \\eta\\,\\gamma_{j(x)}$",
    ]
    colors = [(LIGHT, BLACK), (OCHRE + "33", OCHRE), (BLUE + "33", BLUE), (BLUE + "33", BLUE), (LIGHT, BLACK)]
    flow(ax_flow, texts, colors, [0.6, 1.05, 0.85, 1.55, 1.15], "один шаг градиентного бустинга",
         "следующий шаг: $t \\leftarrow t + 1$; дерево задает области, значения в них — исходная потеря")

    ax_table.set_xlim(0, 10)
    ax_table.set_ylim(0, 4)
    ax_table.axis("off")
    ax_table.text(0.2, 3.5, "функция потерь", fontsize=12, weight="bold")
    ax_table.text(3.2, 3.5, "псевдоостаток $r$", fontsize=12, weight="bold", ha="center")
    ax_table.text(7.2, 3.5, "значение листа $\\gamma$", fontsize=12, weight="bold", ha="center")
    rows = [
        ("MSE, $\\frac{1}{2}(y - a)^2$", "$y - a$ — остаток", "среднее остатков листа"),
        ("MAE, $|y - a|$", "$\\mathrm{sign}(y - a)$ — знак остатка", "медиана остатков листа"),
        ("логистическая, $y \\in \\{0, 1\\}$", "$y - p$, $p = \\sigma(a)$",
         "шаг Ньютона $\\sum (y_i - p_i) \\,/\\, \\sum p_i(1 - p_i)$"),
    ]
    for row, (loss_name, residual_text, leaf_text) in enumerate(rows):
        y = 2.6 - 0.95 * row
        ax_table.text(0.2, y, loss_name, fontsize=12, va="center")
        ax_table.text(3.2, y, residual_text, fontsize=12, va="center", ha="center")
        ax_table.text(7.2, y, leaf_text, fontsize=12, va="center", ha="center")
        ax_table.axhline(y + 0.47, color=GREY, lw=0.6)

    plt.tight_layout()
    plt.show()


plot_gb_scheme()

Дерево отвечает за то, *где* сдвигать модель: оно делит объекты
на области по псевдоостаткам. Исходная функция потерь отвечает за то, *насколько*
сдвигать модель в каждой области.

**Шаги в пространстве прогнозов.** Спуск по предсказаниям виден на игрушечном примере: три объекта $x = 1, 2, 3$ с ответами
$y = (1, 4, 2)$ и бустинг на пнях с $\eta = 1$. Прогнозы модели на обучающей выборке — точка
$\big(a(x_1), a(x_2), a(x_3)\big)$, ответ пня на трех объектах $\big(b_t(x_1), b_t(x_2), b_t(x_3)\big)$ — стрелка,
которая сдвигает эту точку. Минимум MSE — точка $a = y$, остатки $r = y - a$ — стрелка из текущей
точки прямо в минимум.

Пень делит три объекта одним из двух способов: пень A — вопрос $x < 1.5$, листья $\{x_1\}$
и $\{x_2, x_3\}$; пень B — вопрос $x < 2.5$, листья $\{x_1, x_2\}$ и $\{x_3\}$. В листе пень предсказывает
среднее остатков, поэтому сумма его ответов равна сумме остатков:

$$\sum_{i=1}^{3} b_t(x_i) = \sum_{j} |R_j| \cdot \frac{1}{|R_j|} \sum_{x_i \in R_j} r_{ti} = \sum_{i=1}^{3} r_{ti}.$$

На старте сумма остатков равна нулю: $\sum_i r_{1i} = \sum_i y_i - 3\bar{y} = 0$. Тогда и сумма ответов
первого пня равна нулю, и после шага сумма остатков снова равна нулю, и так на каждом шаге. Сумма
прогнозов остается равной $\sum_i y_i = 7$, и прогноз на третьем объекте задают два других:
$a(x_3) = 7 - a(x_1) - a(x_2)$. Все шаги видны на плоскости $\big(a(x_1), a(x_2)\big)$.

С текущими остатками $r_1, r_2, r_3$, для которых $r_1 + r_2 + r_3 = 0$, ответы пней:

- пень A: $\big(r_1,\ \frac{r_2 + r_3}{2},\ \frac{r_2 + r_3}{2}\big) = r_1 \cdot \big(1,\ -\frac{1}{2},\ -\frac{1}{2}\big)$, поскольку $r_2 + r_3 = -r_1$;
- пень B: $\big(\frac{r_1 + r_2}{2},\ \frac{r_1 + r_2}{2},\ r_3\big) = -\frac{r_3}{2} \cdot (1,\ 1,\ -2)$, поскольку $r_1 + r_2 = -r_3$.

На плоскости $\big(a(x_1), a(x_2)\big)$ пень A сдвигает точку только вдоль прямой с наклоном $-\frac{1}{2}$,
пень B — только вдоль диагонали. Ответы шести пней:

In [ ]:
x_toy = np.array([[1.0], [2.0], [3.0]])
y_toy = np.array([1.0, 4.0, 2.0])

toy_boosting = GradientBoostingRegressor(n_estimators=6, learning_rate=1.0, max_depth=1, random_state=SEED)
toy_boosting.fit(x_toy, y_toy)

# прогнозы на трех объектах до первого шага и после каждого шага
toy_start = np.full(3, y_toy.mean())
toy_stages = np.array(list(toy_boosting.staged_predict(x_toy)))
toy_path = np.vstack([toy_start, toy_stages])

# ответ каждого пня на трех объектах и его вопрос
toy_stumps = toy_boosting.estimators_[:, 0]
toy_answers = np.array([stump.predict(x_toy) for stump in toy_stumps])
toy_thresholds = [stump.tree_.threshold[0] for stump in toy_stumps]
toy_answer_sums = toy_answers.sum(axis=1).round(10) + 0.0     # округление убирает ошибку порядка 1e-16
toy_mse = ((toy_stages - y_toy) ** 2).mean(axis=1)

toy_table = pd.DataFrame({
    "пень": ["A: x < 1.5" if threshold < 2 else "B: x < 2.5" for threshold in toy_thresholds],
    "ответ на x₁": toy_answers[:, 0],
    "ответ на x₂": toy_answers[:, 1],
    "ответ на x₃": toy_answers[:, 2],
    "сумма трех ответов": toy_answer_sums,
    "MSE после шага": toy_mse,
}, index=pd.RangeIndex(1, len(toy_stumps) + 1, name="шаг t"))
toy_start_mse = ((toy_start - y_toy) ** 2).mean()
print(f"MSE начального приближения: {toy_start_mse:.3f}")
show_table(toy_table)

In [ ]:
def plot_prediction_space(path, thresholds, x_train, y_train):
    """Бустинг на трех объектах: прогнозы на x₁ и x₂ — точка плоскости, ответ пня — стрелка."""
    total = y_train.sum()
    start, finish = path[0], path[-1]
    colors = [BLUE if threshold < x_train[1, 0] else OCHRE for threshold in thresholds]
    path_mse = ((path - y_train) ** 2).mean(axis=1)

    fig, (ax_data, ax_plane) = plt.subplots(1, 2, figsize=(15, 6.4), gridspec_kw={"width_ratios": [1, 1.45]})

    # --- данные и два возможных пня ---
    split_a = (x_train[0, 0] + x_train[1, 0]) / 2
    split_b = (x_train[1, 0] + x_train[2, 0]) / 2
    ax_data.axvline(split_a, color=BLUE, lw=2, ls=":", label=f"пень A: вопрос x < {split_a:g}")
    ax_data.axvline(split_b, color=OCHRE, lw=2, ls=":", label=f"пень B: вопрос x < {split_b:g}")
    ax_data.axhline(start[0], color=GREY, lw=1.5, ls="--", label=f"$a_0 = \\bar{{y}} = {start[0]:.2f}$")
    edges = [x_train[0, 0] - 0.5, split_a, split_b, x_train[2, 0] + 0.5]
    for number in range(3):
        label = f"сумма после {len(thresholds)} шагов" if number == 0 else None
        ax_data.hlines(finish[number], edges[number], edges[number + 1], color=BLACK, lw=2.5, label=label)
    ax_data.scatter(x_train[:, 0], y_train, s=90, color=BLACK, zorder=3)
    for number, (x_value, y_value) in enumerate(zip(x_train[:, 0], y_train), start=1):
        ax_data.annotate(f"$x_{number}$, $y_{number} = {y_value:g}$", (x_value, y_value), xytext=(8, 8),
                         textcoords="offset points", fontsize=12)
    ax_data.set_xlim(edges[0], edges[-1])
    ax_data.set_ylim(0, y_train.max() + 1.2)
    ax_data.set_xlabel("$x$")
    ax_data.set_ylabel("$y$")
    ax_data.set_title("три объекта и два возможных пня")
    ax_data.legend(loc="upper left", fontsize=10)

    # --- плоскость прогнозов: a(x₁), a(x₂); a(x₃) = сумма − a(x₁) − a(x₂) ---
    low_1, high_1 = min(path[:, 0].min(), y_train[0]) - 0.7, max(path[:, 0].max(), y_train[0]) + 0.7
    low_2, high_2 = min(path[:, 1].min(), y_train[1]) - 0.7, max(path[:, 1].max(), y_train[1]) + 0.5
    first, second = np.meshgrid(np.linspace(low_1, high_1, 400), np.linspace(low_2, high_2, 400))
    third = total - first - second
    loss = ((y_train[0] - first) ** 2 + (y_train[1] - second) ** 2 + (y_train[2] - third) ** 2) / 3
    labeled_levels = np.unique(np.concatenate([path_mse[:3], [2.5, 4.0]]))
    ax_plane.contourf(first, second, loss, levels=30, cmap="Greys", alpha=0.12)
    contours = ax_plane.contour(first, second, loss, levels=labeled_levels, colors=GREY, linewidths=1)
    ax_plane.clabel(contours, fmt=lambda value: f"MSE {value:.2f}", fontsize=9)
    ax_plane.contour(first, second, loss, levels=np.sort(path_mse[3:5]), colors=GREY, linewidths=1)

    # прямые, вдоль которых пни A и B могут сдвинуть стартовую точку
    for direction, color in [((1.0, -0.5), BLUE), ((1.0, 1.0), OCHRE)]:
        direction = np.array(direction)
        line = np.array([start[:2] - 3 * direction, start[:2] + 3 * direction])
        ax_plane.plot(line[:, 0], line[:, 1], color=color, lw=1, ls=":", alpha=0.8)

    for step, (begin, end, color) in enumerate(zip(path[:-1], path[1:], colors), start=1):
        length = np.hypot(*(end[:2] - begin[:2]))
        ax_plane.annotate("", xy=end[:2], xytext=begin[:2], zorder=4,
                          arrowprops=dict(arrowstyle="-|>", color=color, lw=2.4,
                                          mutation_scale=min(20, 8 + 30 * length), shrinkA=0, shrinkB=0))
        if length > 0.2:
            middle = (begin[:2] + end[:2]) / 2
            normal = np.array([-(end[1] - begin[1]), end[0] - begin[0]]) / length
            ax_plane.annotate(f"$b_{step}$", middle, xytext=18 * normal, textcoords="offset points",
                              fontsize=13, color=color, ha="center", va="center", weight="bold")

    ax_plane.annotate("", xy=finish[:2], xytext=start[:2], zorder=3,
                      arrowprops=dict(arrowstyle="-|>", color=BLACK, lw=2, ls="--", mutation_scale=20,
                                      shrinkA=0, shrinkB=0))
    sum_label = "b_1 + b_2 + \\dots + b_{" + str(len(thresholds)) + "}"
    middle = (start[:2] + finish[:2]) / 2
    ax_plane.annotate(f"сумма ответов\n${sum_label}$", middle, xytext=(14, -6), textcoords="offset points",
                      fontsize=12, color=BLACK)
    ax_plane.scatter(*start[:2], s=90, color=BLACK, zorder=5)
    ax_plane.annotate(f"старт $a_0 = ({start[0]:.2f},\\ {start[1]:.2f})$", start[:2], xytext=(10, -16),
                      textcoords="offset points", fontsize=11)
    ax_plane.scatter(y_train[0], y_train[1], s=260, marker="*", color=BLACK, zorder=5)
    ax_plane.annotate(f"минимум: $a = y$, $({y_train[0]:g},\\ {y_train[1]:g})$", (y_train[0], y_train[1]),
                      xytext=(12, 8), textcoords="offset points", fontsize=11)
    ax_plane.plot([], [], color=BLUE, lw=2.4, label="ответ пня A: сдвиг вдоль наклона $-\\frac{1}{2}$")
    ax_plane.plot([], [], color=OCHRE, lw=2.4, label="ответ пня B: сдвиг вдоль диагонали")
    ax_plane.plot([], [], color=BLACK, lw=2, ls="--", label="сумма ответов = остатки на старте")
    ax_plane.legend(loc="lower right", fontsize=10)
    ax_plane.set_xlim(low_1, high_1)
    ax_plane.set_ylim(low_2, high_2)
    ax_plane.set_aspect("equal")
    ax_plane.set_xlabel("прогноз на первом объекте $a(x_1)$")
    ax_plane.set_ylabel("прогноз на втором объекте $a(x_2)$")
    ax_plane.set_title(f"прогнозы на обучающей выборке; $a(x_3) = {total:g} - a(x_1) - a(x_2)$")
    plt.tight_layout()
    plt.show()


plot_prediction_space(toy_path, toy_thresholds, x_toy, y_toy)

Ни один пень не смотрит в минимум: пень A меняет прогноз на $x_1$ только вместе с прогнозами
на $x_2$ и $x_3$, пень B меняет прогнозы на $x_1$ и $x_2$ только вместе. На каждом шаге из двух пней
выбирают тот, что сильнее уменьшает MSE, и он доходит вдоль своей прямой до точки, ближайшей
к минимуму: стрелка заканчивается там, где ее прямая касается линии уровня. В точке касания
второй такой же пень ничего не улучшит, поэтому пни чередуются: A, B, A, B, …

Сумма ответов — пунктирная стрелка — смотрит туда же, куда остатки на старте, прямо в минимум:
$a_0 + b_1 + \dots + b_6 \approx y$. По таблице MSE падает с 1.556 до 0.667 за первый шаг и затем
вчетверо за каждый шаг: каждый пень убирает только ту часть остатка, которая направлена вдоль
его прямой.

В настоящей задаче объектов $n$, прогнозы — точка в $\mathbb{R}^n$, остатки — антиградиент MSE в этой
точке, а дерево с $J$ листьями сдвигает ее только на вектор, постоянный на каждом листе. Дерево
приближает антиградиент таким вектором, и отдельный шаг смотрит мимо минимума, а сумма многих
шагов приходит туда, куда не может прийти ни одно дерево.

## 3. Функции потерь регрессии

Для каждой потери нужны три вещи: псевдоостаток $r = -\partial L/\partial a$, начальная константа
$a_0$ и значение листа $\gamma$. Обозначение: $r_i = y_i - a_i$ — остаток.

**MSE**, $L = \frac{1}{2}(y - a)^2$. Производная $\partial L/\partial a = -(y - a)$, псевдоостаток $r = y - a$.
Константа: $\frac{d}{dc}\sum_i \frac{1}{2}(y_i - c)^2 = -\sum_i (y_i - c) = nc - \sum_i y_i = 0$, $a_0 = \bar{y}$.
Для квадратичной ошибки значение листа

$$\gamma = \arg\min_\gamma \sum_{x_i \in R_j} \tfrac{1}{2}(y_i - a_i - \gamma)^2 = \arg\min_\gamma \sum_{x_i \in R_j} \tfrac{1}{2}(r_i - \gamma)^2.$$

Производная по $\gamma$ равна $-\sum_{x_i \in R_j}(r_i - \gamma) = |R_j|\,\gamma - \sum_{x_i \in R_j} r_i$ и обращается в ноль при
$\gamma = \frac{1}{|R_j|}\sum_{x_i \in R_j} r_i$ — среднем остатков листа, то есть том значении, которое дерево уже предсказывает.
Поэтому в бустинге на остатках (раздел 1) значения листьев не пересчитывают.


**MAE**, $L = |y - a|$. При $y \ne a$ производная $\partial L/\partial a = -\operatorname{sign}(y - a)$, псевдоостаток
$r = \operatorname{sign}(y - a)$. Функция $\sum_i |y_i - c|$ кусочно-линейна; ее производная в точке $c$,
не совпадающей ни с одним $y_i$, равна

$$\frac{d}{dc}\sum_i |y_i - c| = \sum_i -\operatorname{sign}(y_i - c) = \#\{i: y_i < c\} - \#\{i: y_i > c\}.$$

Она отрицательна, пока ниже $c$ меньше половины точек, и положительна, когда больше.
Минимум — в точке, где выше и ниже поровну: $a_0$ — медиана ответов, $\gamma$ — медиана
остатков листа. Псевдоостаток $\pm 1$ не зависит от величины ошибки, поэтому дерево
делит объекты по знаку ошибки, а выбросы влияют на разбиение так же, как обычные точки.

**Функция Хьюбера**, квадратичная при $|y - a| \le \delta$ и линейная дальше:

$$L_\delta = \begin{cases} \frac{1}{2}(y - a)^2, & |y - a| \le \delta, \\ \delta\big(|y - a| - \frac{\delta}{2}\big), & |y - a| > \delta; \end{cases} \qquad r = \begin{cases} y - a, & |y - a| \le \delta, \\ \delta\operatorname{sign}(y - a), & |y - a| > \delta. \end{cases}$$

На границе $|y - a| = \delta$ обе ветви дают $\frac{1}{2}\delta^2$, а обе производные — $\pm\delta$: функция непрерывно дифференцируема.
Значение листа в явном виде не выражается; sklearn берет одношаговое приближение
Фридмана (Friedman, 2001, раздел 4.4; здесь без вывода) $\gamma = \tilde{r} + \frac{1}{k}\sum_i \operatorname{sign}(r_i - \tilde{r})\min(\delta, |r_i - \tilde{r}|)$, где $\tilde{r}$ — медиана
остатков листа, а $\delta$ на каждом шаге — квантиль уровня `alpha` = 0.9 модулей остатков.

**Квантильная потеря** (функция потерь пинбола, pinball loss) уровня $\alpha \in (0, 1)$:

$$L_\alpha(y, a) = \begin{cases} \alpha\,(y - a), & y \ge a, \\ (1 - \alpha)(a - y), & y < a; \end{cases} \qquad r = \begin{cases} \alpha, & y > a, \\ \alpha - 1, & y < a. \end{cases}$$

Производная суммы по константе $c$: $\frac{d}{dc}\sum_i L_\alpha(y_i, c) = -\alpha\,\#\{y_i > c\} + (1 - \alpha)\,\#\{y_i < c\}$.
Если ни один $y_i$ не равен $c$ и доля точек ниже $c$ равна $q$, производная равна
$n\big(-\alpha(1 - q) + (1 - \alpha)q\big) = n(q - \alpha)$ и обращается в ноль при $q = \alpha$: минимум
достигается на квантиле уровня $\alpha$. При $\alpha = 0.5$ потеря равна половине MAE, и квантиль —
медиана. Бустинг с квантильной потерей предсказывает условный квантиль $y$
при данном $x$.

Сводка:

| функция потерь | $L(y, a)$ | псевдоостаток $r$ | $a_0$ | значение листа $\gamma$ |
|:--|:--|:--|:--|:--|
| MSE | $\frac{1}{2}(y - a)^2$ | $y - a$ | среднее $y$ | среднее остатков |
| MAE | $\lvert y - a \rvert$ | $\operatorname{sign}(y - a)$ | медиана $y$ | медиана остатков |
| функция Хьюбера | $L_\delta(y, a)$ | остаток, обрезанный на уровне $\delta$ | медиана $y$ | приближенный минимум |
| квантильная | $L_\alpha(y, a)$ | $\alpha$ или $\alpha - 1$ | квантиль $\alpha$ ответов | квантиль $\alpha$ остатков |

Устойчивость к выбросам: шесть ответов обучающей выборки сдвигают вверх на 4.

In [ ]:
rng_outliers = np.random.default_rng(3)
outlier_index = rng_outliers.choice(len(y_sine), 6, replace=False)

y_outliers = y_sine.copy()
y_outliers[outlier_index] = y_outliers[outlier_index] + 4          # шесть ответов сдвинуты вверх на 4

print("номера выбросов:", np.sort(outlier_index))

Первый шаг бустинга на этих данных с MSE и с MAE. Стрелка — псевдоостаток объекта: куда
и насколько шаг градиентного спуска сдвинул бы предсказание в этой точке. Пень обучают
на стрелках по MSE, затем значения его листьев пересчитывают по исходной потере.

In [ ]:
fig, (ax_mse, ax_mae) = plt.subplots(1, 2, figsize=(18, 5.8), sharey=True)
x_points = x_sine[:, 0]

# --- MSE: стрелка от модели до точки, длина — остаток ---
start_mse = y_outliers.mean()                                    # a_0 для MSE — среднее
pseudo_mse = y_outliers - start_mse                              # псевдоостаток — сам остаток

stump_mse = DecisionTreeRegressor(max_depth=1, random_state=SEED)
stump_mse.fit(x_sine, pseudo_mse)
model_mse = start_mse + stump_mse.predict(x_grid)               # значение листа — среднее остатков

ax_mse.quiver(x_points, np.full(len(x_points), start_mse), np.zeros(len(x_points)), pseudo_mse,
              angles="xy", scale_units="xy", scale=1, width=0.0025, color=OCHRE, alpha=0.8)
ax_mse.scatter(x_points, y_outliers, s=14, color=GREY, zorder=3)
ax_mse.axhline(start_mse, color=BLUE, lw=1.5, ls="--", label=f"$a_0$ = среднее = {start_mse:.2f}")
ax_mse.plot(x_grid[:, 0], model_mse, color=BLACK, lw=2, label="$a_1 = a_0 + \\gamma$, $\\gamma$ — среднее остатков листа")
ax_mse.set_title("MSE: псевдоостаток $y - a$, стрелки разной длины, у выбросов — самые длинные", fontsize=12)
ax_mse.set_xlabel("x")
ax_mse.set_ylabel("y")
ax_mse.legend(loc="upper right", fontsize=10)

# --- MAE: стрелки одной длины, дерево по знакам, значения листьев — медианы остатков ---
start_mae = np.median(y_outliers)                                # a_0 для MAE — медиана
residuals_mae = y_outliers - start_mae
pseudo_mae = np.sign(residuals_mae)                              # псевдоостаток — знак остатка

stump_mae = DecisionTreeRegressor(max_depth=1, random_state=SEED)
stump_mae.fit(x_sine, pseudo_mae)
leaf_of_point = stump_mae.apply(x_sine)
leaf_of_grid = stump_mae.apply(x_grid)

leaf_lines = []
model_mae = np.full(len(x_grid), start_mae)
for leaf in np.unique(leaf_of_point):
    in_leaf = leaf_of_point == leaf
    mean_sign = pseudo_mae[in_leaf].mean()                       # значение, которое предсказывает само дерево
    median_residual = np.median(residuals_mae[in_leaf])          # значение листа после пересчета по MAE
    model_mae[leaf_of_grid == leaf] = start_mae + median_residual
    side = "x ≤ порог" if x_points[in_leaf].max() <= stump_mae.tree_.threshold[0] else "x > порог"
    leaf_lines.append(f"{side}: среднее знаков {mean_sign:+.2f} → медиана остатков {median_residual:+.2f}")

arrow_length = 0.6 * pseudo_mae                                  # одна длина для всех: важен только знак
ax_mae.quiver(x_points, np.full(len(x_points), start_mae), np.zeros(len(x_points)), arrow_length,
              angles="xy", scale_units="xy", scale=1, width=0.0025, color=OCHRE, alpha=0.8)
ax_mae.scatter(x_points, y_outliers, s=14, color=GREY, zorder=3)
ax_mae.axhline(start_mae, color=BLUE, lw=1.5, ls="--", label=f"$a_0$ = медиана = {start_mae:.2f}")
ax_mae.plot(x_grid[:, 0], model_mae, color=BLACK, lw=2, label="$a_1 = a_0 + \\gamma$, $\\gamma$ — медиана остатков листа")
ax_mae.axvline(stump_mae.tree_.threshold[0], color=BLUE, ls=":", lw=1)
ax_mae.text(0.02, 0.04, "\n".join(leaf_lines), transform=ax_mae.transAxes, fontsize=10, va="bottom",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=BLUE))
ax_mae.set_title("MAE: псевдоостаток sign(y − a), стрелки одной длины", fontsize=12)
ax_mae.set_xlabel("x")
ax_mae.legend(loc="upper right", fontsize=10)

plt.tight_layout()
plt.show()

print(f"MSE: порог пня {stump_mse.tree_.threshold[0]:.2f}, значения листьев {stump_mse.tree_.value[[1, 2], 0, 0].round(2)}")
print(f"MAE: порог пня {stump_mae.tree_.threshold[0]:.2f}")

С MSE стрелки выбросов в несколько раз длиннее остальных, и пень ставит порог так,
чтобы отделить участок с выбросами: вопрос «$x \le 2.45$?», слева прибавляет 0.94, и модель
на всем участке поднимается до 1.21 — выше любого значения синуса. С MAE у каждого объекта
стрелка одной длины: пень учитывает только, выше или ниже модели лежит точка, и ставит
порог 2.97 почти там, где синус меняет знак ($\pi \approx 3.14$). Само дерево предсказывает в листе
среднее знаков (+0.95 и −0.81), а значение листа пересчитывают по MAE — медиану остатков
листа (+0.62 и −0.69), на которую шесть выбросов почти не влияют.

Бустинг с тремя функциями потерь на тех же данных, 300 пней, $\eta = 0.1$; качество — MSE
на тестовом сете без выбросов:

In [ ]:
robust_losses = {"squared_error": ("MSE", BLUE), "absolute_error": ("MAE", BLACK), "huber": ("функцией Хьюбера", OCHRE)}
true_curve = np.sin(x_grid[:, 0])

plt.figure(figsize=(11, 5))
plt.scatter(x_sine[:, 0], y_outliers, s=14, color=GREY, label="обучающая выборка")
plt.scatter(x_sine[outlier_index, 0], y_outliers[outlier_index], s=60, facecolor="none", edgecolor="red", lw=1.5, label="выбросы")
plt.plot(x_grid[:, 0], true_curve, color=GREY, lw=1.5, ls="--", label="sin x")

for loss, (label, color) in robust_losses.items():
    model = GradientBoostingRegressor(loss=loss, n_estimators=300, learning_rate=0.1, max_depth=1, random_state=SEED)
    model.fit(x_sine, y_outliers)

    robust_prediction = model.predict(x_sine_test)
    robust_test_mse = np.mean((robust_prediction - y_sine_test) ** 2)
    curve = model.predict(x_grid)
    plt.plot(x_grid[:, 0], curve, color=color, lw=2, label=f"бустинг с {label}: MSE на тесте {robust_test_mse:.3f}")

plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

MSE на тестовом сете: 0.540 с MSE, 0.101 с MAE, 0.095 с функцией Хьюбера; бустинг с MSE
на данных без выбросов дает 0.082.
С MSE псевдоостатки выбросов близки к 4, разбиения пней выделяют участки с выбросами,
и модель приближается к выбросам вместе с соседними точками. У MAE псевдоостаток выброса $+1$, а медиана в листе на выбросы почти
не реагирует; функция Хьюбера ограничивает псевдоостатки выбросов уровнем $\delta$.

**Квантильная регрессия.** Данные с шумом, дисперсия которого растет с $x$:
$y = \sin x + (0.1 + 0.1x)\,\xi$, $\xi \sim \mathcal{N}(0, 1)$, $x \in [0, 10]$. Бустинг с квантильными потерями уровней
0.05, 0.5 и 0.95 дает медиану и 90-процентный интервал предсказания; доля новых
точек внутри интервала должна быть около 0.9.

In [ ]:
rng_quantile = np.random.default_rng(1)


def make_heteroscedastic(n):
    x = np.sort(rng_quantile.uniform(0, 10, n))[:, None]
    noise_scale = 0.1 + 0.1 * x[:, 0]
    y = np.sin(x[:, 0]) + noise_scale * rng_quantile.normal(0, 1, n)
    data = (x, y)
    return data


x_hetero, y_hetero = make_heteroscedastic(500)
x_hetero_test, y_hetero_test = make_heteroscedastic(5000)
x_hetero_grid = np.linspace(0, 10, 400)[:, None]

quantile_curves = {}
quantile_test = {}
for alpha in [0.05, 0.5, 0.95]:
    model = GradientBoostingRegressor(loss="quantile", alpha=alpha, n_estimators=200, learning_rate=0.1,
                                      max_depth=2, random_state=SEED)
    model.fit(x_hetero, y_hetero)
    quantile_curves[alpha] = model.predict(x_hetero_grid)
    quantile_test[alpha] = model.predict(x_hetero_test)

below_lower = np.mean(y_hetero_test < quantile_test[0.05])
above_upper = np.mean(y_hetero_test > quantile_test[0.95])
inside = 1 - below_lower - above_upper

plt.figure(figsize=(10, 4.8))
plt.scatter(x_hetero[:, 0], y_hetero, s=8, color=GREY, label="обучающая выборка")
plt.fill_between(x_hetero_grid[:, 0], quantile_curves[0.05], quantile_curves[0.95], color=BLUE, alpha=0.2,
                 label="между квантилями 0.05 и 0.95")
plt.plot(x_hetero_grid[:, 0], quantile_curves[0.5], color=BLACK, lw=2, label="квантиль 0.5 (медиана)")
plt.plot(x_hetero_grid[:, 0], np.sin(x_hetero_grid[:, 0]), color=OCHRE, lw=1.5, ls="--", label="sin x")
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="lower left")
plt.show()

print(f"тестовый сет: ниже квантиля 0.05 — {below_lower:.3f}, выше квантиля 0.95 — {above_upper:.3f}, внутри — {inside:.3f}")

На тестовом сете ниже квантиля 0.05 оказались 6.4% точек, выше квантиля 0.95 — 6.9%,
внутри интервала — 86.8% при ожидаемых 90%: по 500 объектам крайние квантили
оцениваются с ошибкой, и интервал немного уже нужного. Ширина интервала растет с $x$
вместе с разбросом шума; бустинг с MSE дал бы только условное среднее.

## 4. Бинарная классификация

Ответы $y \in \{0, 1\}$, модель выдает оценку $a(x)$, вероятность класса 1 — $p = \sigma(a) = \frac{1}{1 + e^{-a}}$.
Логистическая потеря

$$L(y, a) = -y\ln p - (1 - y)\ln(1 - p).$$

Подстановка $\ln p = -\ln(1 + e^{-a})$ и $\ln(1 - p) = \ln\frac{e^{-a}}{1 + e^{-a}} = -a - \ln(1 + e^{-a})$ дает
$L = \ln(1 + e^{-a}) + (1 - y)\,a = \ln(1 + e^{a}) - y\,a$; последний переход —
$\ln(1 + e^{-a}) + a = \ln(e^{a} + 1)$.

**Первая производная.** Из последней формы:

$$\frac{\partial L}{\partial a} = \frac{e^{a}}{1 + e^{a}} - y = \sigma(a) - y = p - y, \qquad r = y - p.$$

**Вторая производная** — производная $p - y$ по $a$, то есть производная сигмоиды:

$$\frac{\partial^2 L}{\partial a^2} = \frac{dp}{da} = \frac{e^{-a}}{(1 + e^{-a})^2} = p(1 - p).$$

Она положительна, и $L$ выпукла по $a$; наибольшая кривизна $\frac{1}{4}$ — при $p = 0.5$.

**Начальное приближение.** $\frac{d}{dc}\sum_i L(y_i, c) = \sum_i (\sigma(c) - y_i) = 0$ при $\sigma(c) = \bar{y}$,
$a_0 = \ln\frac{\bar{y}}{1 - \bar{y}}$ — логарифм отношения шансов.

**Значение листа.** Уравнение $\sum_{x_i \in R}\big(\sigma(a_i + \gamma) - y_i\big) = 0$ не решается в явном виде.
Разложение суммы потерь листа в ряд Тейлора до второго порядка по $\gamma$ с $g_i = p_i - y_i$
и $h_i = p_i(1 - p_i)$:

$$\sum_{x_i \in R} L(y_i, a_i + \gamma) \approx \sum_{x_i \in R} L(y_i, a_i) + \gamma \sum_{x_i \in R} g_i + \frac{\gamma^2}{2}\sum_{x_i \in R} h_i.$$

Квадратичная функция $\gamma G + \frac{1}{2}\gamma^2 H$ с $H > 0$ минимальна при $G + \gamma H = 0$:

$$\gamma = -\frac{G}{H} = \frac{\sum_{x_i \in R}(y_i - p_i)}{\sum_{x_i \in R} p_i(1 - p_i)}.$$

Это один шаг метода Ньютона из точки $\gamma = 0$. При той же $G$ шаг тем больше по модулю,
чем меньше $H$: малая вторая производная означает малую кривизну, квадратичное
приближение почти линейно, и его минимум далеко от нуля. Малая $H$ бывает, когда
вероятности в листе близки к 0 или 1. Схема одного шага для классификации:

In [ ]:
fig, ax = plt.subplots(figsize=(18, 3.6))
ax.axis("off")
ax.set_title("один шаг градиентного бустинга для классификации", fontsize=14, loc="left")

class_steps = [
    "оценка\n$a_{t-1}(x)$",
    "вероятность класса 1\n$p_i = \\sigma(a_{t-1}(x_i))$",
    "псевдоостатки\n$r_i = y_i - p_i$",
    "дерево по MSE\nна парах $(x_i, r_i)$",
    "шаг Ньютона в листе\n$\\gamma_j = \\sum (y_i - p_i) \\,/\\, \\sum p_i(1 - p_i)$",
    "$a_t(x) = a_{t-1}(x) + \\eta\\,\\gamma_{j(x)}$",
]
class_colors = [("#EEF2F6", BLACK), ("#EEF2F6", BLACK), (OCHRE + "33", OCHRE), (BLUE + "33", BLUE),
                (BLUE + "33", BLUE), ("#EEF2F6", BLACK)]
class_half_widths = [0.38, 0.83, 0.62, 0.62, 1.1, 1.0]       # половина ширины блока в единицах оси
gap = 0.55

class_x = []
cursor = 0.1
for half_width in class_half_widths:
    class_x.append(cursor + half_width)
    cursor = cursor + 2 * half_width + gap
ax.set_xlim(0, cursor - gap + 0.1)
ax.set_ylim(0, 3)

for x, text, (face, edge) in zip(class_x, class_steps, class_colors):
    box(ax, x, 1.9, text, color=face, edge=edge, size=11)
for left in range(len(class_x) - 1):
    start = class_x[left] + class_half_widths[left] + 0.05
    end = class_x[left + 1] - class_half_widths[left + 1] - 0.05
    arrow(ax, (start, 1.9), (end, 1.9))
arrow(ax, (class_x[-1], 1.25), (class_x[0], 1.25), bend=-0.06)
middle = (cursor - gap) / 2
ax.text(middle, 0.15, "объект, который модель уже уверенно классифицирует верно, дает $y_i - p_i \\approx 0$ и $p_i(1 - p_i) \\approx 0$",
        ha="center", fontsize=11, color=GREY)
plt.show()

Классификация на одной оси: 100 точек, класс 1 вероятнее в середине отрезка $[0, 2\pi]$.
Четыре шага бустинга пней с $\eta = 1$. Сверху — вероятность класса 1 до шага и остатки $y - p$
(серые отрезки от кривой до меток 0 и 1), в середине — остатки точками и пень, обученный
на них, с шагами Ньютона в листьях; снизу — вероятность после шага.

In [ ]:
rng_classes = np.random.default_rng(4)
x_classes = np.sort(rng_classes.uniform(0, 2 * np.pi, 100))[:, None]
true_probability = expit(4 * np.cos(x_classes[:, 0] - np.pi))        # класс 1 вероятнее в середине оси
y_classes = (rng_classes.random(100) < true_probability).astype(int)

share_positive = y_classes.mean()
initial_score = np.log(share_positive / (1 - share_positive))      # a_0 — логарифм отношения шансов

scores_train = np.full(len(y_classes), initial_score)
scores_grid = np.full(len(x_grid), initial_score)
class_history = []                                                 # что нужно для рисунка на каждом шаге

for step in range(4):
    probability = expit(scores_train)
    residuals = y_classes - probability                            # псевдоостатки y − p

    stump = DecisionTreeRegressor(max_depth=1, random_state=SEED)
    stump.fit(x_classes, residuals)
    leaf_of_point = stump.apply(x_classes)
    leaf_of_grid = stump.apply(x_grid)

    newton_values = {}
    for leaf in np.unique(leaf_of_point):
        in_leaf = leaf_of_point == leaf
        numerator = residuals[in_leaf].sum()
        denominator = np.sum(probability[in_leaf] * (1 - probability[in_leaf]))
        newton_values[leaf] = numerator / denominator

    step_train = np.array([newton_values[leaf] for leaf in leaf_of_point])
    step_grid = np.array([newton_values[leaf] for leaf in leaf_of_grid])
    new_scores_train = scores_train + step_train                   # η = 1
    new_scores_grid = scores_grid + step_grid

    old_probability = expit(scores_train)
    new_probability = expit(new_scores_train)
    old_loss = -np.mean(y_classes * np.log(old_probability) + (1 - y_classes) * np.log(1 - old_probability))
    new_loss = -np.mean(y_classes * np.log(new_probability) + (1 - y_classes) * np.log(1 - new_probability))
    new_accuracy = np.mean((new_probability > 0.5) == y_classes)

    class_history.append({
        "residuals": residuals, "stump": stump, "newton": newton_values,
        "old_grid": expit(scores_grid), "new_grid": expit(new_scores_grid), "old_train": old_probability,
        "old_loss": old_loss, "new_loss": new_loss, "accuracy": new_accuracy,
    })
    scores_train = new_scores_train
    scores_grid = new_scores_grid

point_colors = np.where(y_classes == 1, OCHRE, BLUE)
fig, axes = plt.subplots(3, 4, figsize=(20, 12), sharex=True)

for column, record in enumerate(class_history):
    step = column + 1
    stump = record["stump"]
    threshold = stump.tree_.threshold[0]
    left_leaf, right_leaf = stump.tree_.children_left[0], stump.tree_.children_right[0]
    rule = (f"x ≤ {threshold:.2f}?\nда: $\\gamma$ = {record['newton'][left_leaf]:+.2f}\n"
            f"нет: $\\gamma$ = {record['newton'][right_leaf]:+.2f}")

    ax = axes[0, column]
    ax.vlines(x_classes[:, 0], record["old_train"], y_classes, color=GREY, lw=0.8, alpha=0.7)
    ax.scatter(x_classes[:, 0], y_classes, s=14, c=point_colors, zorder=3)
    ax.plot(x_grid[:, 0], record["old_grid"], color=BLACK, lw=2, label=f"$p_{step - 1}(x)$")
    ax.set_title(f"шаг {step}: вероятность $p_{step - 1}$ и остатки $y - p$", fontsize=12)
    ax.set_ylim(-0.1, 1.1)
    ax.legend(loc="center right")

    ax = axes[1, column]
    ax.axhline(0, color=GREY, lw=0.8)
    ax.scatter(x_classes[:, 0], record["residuals"], s=14, c=point_colors, zorder=3)
    ax.plot(x_grid[:, 0], stump.predict(x_grid), color=BLUE, lw=2.5, label="пень: среднее $y - p$ в листе")
    ax.axvline(threshold, color=BLUE, ls=":", lw=1)
    ax.text(0.97, 0.96, rule, transform=ax.transAxes, ha="right", va="top", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=BLUE))
    ax.set_title(f"пень $b_{step}$ на остатках $y - p$", fontsize=12)
    ax.set_ylim(-1.05, 1.05)
    ax.legend(loc="lower left", fontsize=9)

    ax = axes[2, column]
    ax.scatter(x_classes[:, 0], y_classes, s=14, c=point_colors, zorder=3)
    ax.plot(x_grid[:, 0], record["old_grid"], color=GREY, lw=1.5, ls="--", label=f"$p_{step - 1}$")
    ax.plot(x_grid[:, 0], record["new_grid"], color=BLACK, lw=2, label=f"$p_{step} = \\sigma(a_{step})$")
    ax.axhline(0.5, color=GREY, lw=0.6, ls=":")
    ax.set_title(f"log loss {record['old_loss']:.3f} → {record['new_loss']:.3f}, accuracy {record['accuracy']:.2f}", fontsize=12)
    ax.set_ylim(-0.1, 1.1)
    ax.set_xlabel("x")
    ax.legend(loc="center right")

axes[0, 0].set_ylabel("класс и вероятность")
axes[1, 0].set_ylabel("остаток $y - p$")
axes[2, 0].set_ylabel("вероятность класса 1")
plt.tight_layout()
plt.show()

print(f"доля класса 1: {share_positive:.2f}, a_0 = {initial_score:.3f}")

Первый пень ставит порог 4.41 у правой границы класса 1: слева оценка растет на 0.68,
справа падает на 1.67; accuracy 0.75, левый край оси пока отнесен к классу 1. Второй пень
находит левую границу, 1.40, и опускает оценку левого края на 3.14: accuracy 0.91, log loss
падает с 0.541 до 0.343. Шаг большой, потому что все объекты этого листа класса 0, а модель
давала им $p = 0.68$: $\gamma = \sum(0 - 0.68) / \sum 0.68 \cdot 0.32 = -0.68 / 0.22 \approx -3.1$. Третий и четвертый пни
уточняют края. В верхнем ряду серые отрезки укорачиваются: у объектов, которые модель уже
уверенно классифицирует верно, остаток $y - p$ близок к нулю. Из выборки их не убирают:
к числителю $\sum(y - p)$ и знаменателю $\sum p(1 - p)$ шага Ньютона они почти ничего не добавляют,
но остаются в листе и учитываются при выборе разбиений.

Двумерные данные — две луны из занятия 8, 300 точек. Вероятность класса 1
после 1, 5, 20 и 200 деревьев глубины 2 при $\eta = 0.3$:

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=0)

moons_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.3, max_depth=2, random_state=SEED)
moons_model.fit(X_moons, y_moons)

x_values = np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 250)
y_values = np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 250)
grid_x, grid_y = np.meshgrid(x_values, y_values)
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]           # (62500, 2): все точки сетки

shown_stages = [1, 5, 20, 200]
stage_probabilities = {}
for number, probabilities in enumerate(moons_model.staged_predict_proba(grid_points), start=1):
    if number in shown_stages:
        stage_probabilities[number] = probabilities[:, 1].reshape(grid_x.shape)

point_colors = ListedColormap([BLUE, OCHRE])
probability_colors = LinearSegmentedColormap.from_list("класс 0 — класс 1", [BLUE, "white", OCHRE])
extent = (x_values[0], x_values[-1], y_values[0], y_values[-1])

fig, axes = plt.subplots(1, 4, figsize=(19, 4.6))
for ax, stage in zip(axes, shown_stages):
    background = ax.imshow(stage_probabilities[stage], extent=extent, origin="lower", aspect="auto",
                           cmap=probability_colors, vmin=0, vmax=1, alpha=0.6)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=point_colors, s=12, edgecolor="white", lw=0.4)
    ax.set_title(f"T = {stage}", fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)

plt.tight_layout()
colorbar = fig.colorbar(background, ax=axes, shrink=0.85, pad=0.01)
colorbar.set_label("вероятность класса 1")
plt.show()

После одного дерева при $\eta = 0.3$ вероятности близки к 0.5 и постоянны в четырех
областях, заданных тремя вопросами дерева глубины 2. К 20 деревьям граница повторяет форму лун. К 200
деревьям появляются узкие полосы и островки вокруг отдельных точек чужого
класса: модель подгоняется под шум, как синус в разделе 1.

Проверка формулы на модели sklearn из двух деревьев на лунах: для каждого листа
второго дерева вычисляют $-G/H$ по вероятностям после первого дерева
и сравнивают со значением листа, которое хранит sklearn.

In [ ]:
two_trees = GradientBoostingClassifier(n_estimators=2, learning_rate=1.0, max_depth=2, random_state=SEED)
two_trees.fit(X_moons, y_moons)

staged_scores = list(two_trees.staged_decision_function(X_moons))
scores_after_first = staged_scores[0][:, 0]                    # a_1(x_i): оценки после первого дерева
probability = 1 / (1 + np.exp(-scores_after_first))            # p_i = σ(a_1(x_i))

gradient = probability - y_moons                               # g_i = p_i − y_i
hessian = probability * (1 - probability)                      # h_i = p_i (1 − p_i)

second_tree = two_trees.estimators_[1, 0]
leaf_of_object = second_tree.apply(X_moons)                    # номер листа второго дерева для каждого объекта

newton_rows = []
for leaf in np.unique(leaf_of_object):
    in_leaf = leaf_of_object == leaf

    gradient_sum = gradient[in_leaf].sum()
    hessian_sum = hessian[in_leaf].sum()
    newton_step = -gradient_sum / hessian_sum

    newton_rows.append({
        "лист": leaf,
        "объектов": in_leaf.sum(),
        "сумма g": gradient_sum,
        "сумма h": hessian_sum,
        "−G / H": newton_step,
        "значение листа в sklearn": second_tree.tree_.value[leaf, 0, 0],
    })

newton_table = pd.DataFrame(newton_rows).set_index("лист")
newton_table.round(4)

Значения листьев совпадают с формулой. На рисунке выше вероятность после первого
дерева близка к 0.5 и уточняется с каждым деревом; к 200 деревьям появляются островки
вокруг отдельных точек — переобучение.

**Данные Хасти** (книга The Elements of Statistical Learning, ESL, рисунок 10.2): 10 признаков
$x_j \sim \mathcal{N}(0, 1)$, класс 1 при $\sum_j x_j^2 > 9.34$, медиане $\chi^2_{10}$; 2000 объектов обучения и 10 000 тестового сета.

In [ ]:
Xh, yh_signs = make_hastie_10_2(n_samples=12000, random_state=1)
yh = (yh_signs > 0).astype(int)                     # метки −1 и 1 переводят в 0 и 1

Xh_train, yh_train = Xh[:2000], yh[:2000]           # 2000 объектов обучения
Xh_test, yh_test = Xh[2000:], yh[2000:]             # 10 000 объектов тестового сета

hastie_results = {}


def evaluate_classifier(name, model):
    """Обучить модель на данных Хасти и записать в таблицу долю ошибок на тестовом сете."""
    started = time.perf_counter()
    model.fit(Xh_train, yh_train)
    fit_time = time.perf_counter() - started

    prediction = model.predict(Xh_test)
    error_rate = np.mean(prediction != yh_test)

    hastie_results[name] = {"доля ошибок на тесте": error_rate, "время обучения, с": fit_time}
    print(f"{name}: доля ошибок {error_rate:.3f}, обучение {fit_time:.2f} с")
    return model


print("обучение:", Xh_train.shape, " доля класса 1:", yh_train.mean().round(3))

Пень, дерево без ограничений, случайный лес и бустинг 400 пней при $\eta = 1$;
кривая — доля ошибок бустинга в зависимости от числа пней.

In [ ]:
evaluate_classifier("пень", DecisionTreeClassifier(max_depth=1, random_state=SEED))
full_tree = evaluate_classifier("дерево без ограничений", DecisionTreeClassifier(random_state=SEED))
evaluate_classifier("случайный лес, 300 деревьев", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))
stump_boosting = evaluate_classifier("бустинг 400 пней, η = 1",
                                     GradientBoostingClassifier(n_estimators=400, learning_rate=1.0, max_depth=1, random_state=SEED))

stump_errors = []
for prediction in stump_boosting.staged_predict(Xh_test):
    stump_errors.append(np.mean(prediction != yh_test))
stump_errors = np.array(stump_errors)

tree_error = hastie_results["дерево без ограничений"]["доля ошибок на тесте"]
forest_error = hastie_results["случайный лес, 300 деревьев"]["доля ошибок на тесте"]
beats_tree = int(np.argmax(stump_errors < tree_error)) + 1           # первое T, при котором бустинг лучше
beats_forest = int(np.argmax(stump_errors < forest_error)) + 1
print(f"бустинг пней лучше дерева начиная с T = {beats_tree}, лучше леса начиная с T = {beats_forest}")

plt.figure(figsize=(10, 4.8))
plt.plot(np.arange(1, 401), stump_errors, color=BLUE, lw=2, label="бустинг пней")
reference_lines = [
    ("пень", GREY, "--"),
    ("дерево без ограничений", BLACK, "--"),
    ("случайный лес, 300 деревьев", OCHRE, "--"),
]
for name, color, style in reference_lines:
    error_rate = hastie_results[name]["доля ошибок на тесте"]
    plt.axhline(error_rate, color=color, ls=style, label=f"{name}: {error_rate:.3f}")
plt.xlabel("число деревьев T")
plt.ylabel("доля ошибок на тестовом сете")
plt.ylim(0, 0.5)
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

print("листьев у дерева без ограничений:", full_tree.get_n_leaves())
show_table(pd.DataFrame(hastie_results).T)

Пень ошибается в 0.459 случаев, дерево без ограничений — в 0.243, лес — в 0.138,
бустинг 400 пней — в 0.058. Граница $\sum_j x_j^2 = 9.34$ — сумма функций отдельных признаков,
и бустинг пней, аддитивная модель (раздел 7), описывает ее без взаимодействий.

## 5. Многоклассовая классификация

Для $K$ классов модель хранит $K$ оценок $a_1(x), \dots, a_K(x)$, вероятности — softmax:

$$p_k = \frac{e^{a_k}}{\sum_{m=1}^{K} e^{a_m}}, \qquad L(y, \mathbf{a}) = -\ln p_y = -a_y + \ln\sum_{m=1}^{K} e^{a_m}.$$

**Производная по оценке $a_k$.** Первое слагаемое дает $-[y = k]$, второе — $\frac{e^{a_k}}{\sum_m e^{a_m}} = p_k$:

$$\frac{\partial L}{\partial a_k} = p_k - [y = k], \qquad r_k = [y = k] - p_k.$$

**Вторая производная** по той же оценке — производная частного $p_k = e^{a_k}/S$, $S = \sum_m e^{a_m}$, где $\partial S/\partial a_k = e^{a_k}$:

$$\frac{\partial^2 L}{\partial a_k^2} = \frac{\partial p_k}{\partial a_k} = \frac{e^{a_k} S - e^{a_k} e^{a_k}}{S^2} = p_k - p_k^2 = p_k(1 - p_k).$$

**Начальное приближение.** Для констант $c_1, \dots, c_K$ производная суммы потерь по $c_k$ равна
$\sum_i (p_k - [y_i = k]) = n\,p_k - n_k$, где $n_k$ — число объектов класса $k$; она равна нулю при $p_k = \bar{y}_k = n_k/n$.
Константы $c_k = \ln \bar{y}_k$ дают $p_k = \bar{y}_k / \sum_m \bar{y}_m = \bar{y}_k$, поэтому $a_{0k} = \ln \bar{y}_k$. Прибавление одной константы
ко всем $a_k$ не меняет вероятностей, и оценки определены с точностью до общего сдвига
(sklearn вычитает из $\ln \bar{y}_k$ их среднее).

**Алгоритм.** На итерации $t$ по текущим вероятностям считают псевдоостатки всех классов
и строят $K$ деревьев: дерево $k$ обучают на $r_{ik}$. Значение листа — диагональный шаг
Ньютона с множителем Фридмана:

$$\gamma = \frac{K - 1}{K}\cdot\frac{\sum_{x_i \in R} r_{ik}}{\sum_{x_i \in R} p_{ik}(1 - p_{ik})}.$$

Без множителя это шаг Ньютона по каждой оценке отдельно, как в разделе 4. Множитель $\frac{K - 1}{K}$
Фридман получил для симметричной параметризации, в которой сумма оценок равна нулю
(Friedman, 2001, алгоритм 6; здесь без вывода); в sklearn он оставлен для совместимости
и действует как дополнительный множитель темпа обучения. На $T$ итерациях модель строит $KT$ деревьев. Проверка формулы на винах:
вторая итерация модели sklearn из двух итераций.

In [ ]:
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
n_classes = 3

two_rounds = GradientBoostingClassifier(n_estimators=2, learning_rate=1.0, max_depth=2, random_state=SEED)
two_rounds.fit(X_wine, y_wine)

probabilities_after_first = list(two_rounds.staged_predict_proba(X_wine))[0]      # (178, 3): p_ik после первой итерации
one_hot = np.eye(n_classes)[y_wine]
residuals = one_hot - probabilities_after_first                                  # r_ik = [y_i = k] − p_ik
factor = (n_classes - 1) / n_classes

multiclass_rows = []
for k in range(n_classes):
    tree = two_rounds.estimators_[1, k]
    leaf_of_object = tree.apply(X_wine)

    for leaf in np.unique(leaf_of_object):
        in_leaf = leaf_of_object == leaf
        p_leaf = probabilities_after_first[in_leaf, k]

        numerator = residuals[in_leaf, k].sum()
        denominator = np.sum(p_leaf * (1 - p_leaf))
        formula = factor * numerator / denominator

        multiclass_rows.append({
            "класс": k,
            "лист": leaf,
            "объектов": in_leaf.sum(),
            "по формуле": formula,
            "значение листа в sklearn": tree.tree_.value[leaf, 0, 0],
        })

pd.DataFrame(multiclass_rows).set_index(["класс", "лист"]).round(4)

Все значения совпадают. Многоклассовый бустинг в sklearn обучает $K$ деревьев за итерацию
и для больших $K$ медленный; гистограммные реализации (раздел 10) делают то же быстрее.

## 6. AdaBoost — бустинг с экспоненциальной потерей

**AdaBoost** (adaptive boosting, адаптивный бустинг; Freund, Schapire, 1997) — первый
практический алгоритм бустинга. Позже показали (Friedman, Hastie, Tibshirani, 2000),
что это пошаговое построение суммы классификаторов с **экспоненциальной потерей**.
Метки $y \in \{-1, +1\}$, базовые модели — классификаторы $b(x) \in \{-1, +1\}$, потеря

$$L(y, a) = e^{-y\,a}.$$

Каждый следующий классификатор обучают на тех же объектах, но с **весами** (`sample_weight`):
объекты, на которых ансамбль ошибается, получают больший вес, и классификатор старается
прежде всего классифицировать верно их. Алгоритм:

1. веса объектов $w_i = \frac{1}{n}$;
2. для $t = 1, \dots, T$:
   - классификатор $b_t$, например пень, обучают с весами $w_i$;
   - взвешенная доля ошибок $\text{err}_t = \sum_{i:\, b_t(x_i) \ne y_i} w_i$ (сумма весов равна 1)
     и вес классификатора $\alpha_t = \frac{1}{2}\ln\frac{1 - \text{err}_t}{\text{err}_t}$;
   - веса пересчитывают, $w_i \leftarrow w_i\, e^{-\alpha_t y_i b_t(x_i)}$, и нормируют на сумму 1;
3. ответ — знак взвешенной суммы: $a(x) = \sum_{t=1}^{T} \alpha_t\, b_t(x)$, класс $\operatorname{sign} a(x)$.

Формулы для $\alpha_t$ и весов выводят из экспоненциальной потери.

**Шаг.** На шаге $t$ ищут $\alpha$ и $b$, минимизирующие

$$\sum_{i} e^{-y_i (a_{t-1}(x_i) + \alpha b(x_i))} = \sum_{i} w_i\, e^{-\alpha y_i b(x_i)}, \qquad w_i = e^{-y_i a_{t-1}(x_i)}.$$

Вес $w_i$ не зависит от $\alpha$ и $b$ и тем больше, чем хуже текущая модель на объекте $i$.
Произведение $y_i b(x_i)$ равно $+1$ на верно классифицированных объектах и $-1$ на ошибках:

$$\sum_i w_i\, e^{-\alpha y_i b(x_i)} = e^{-\alpha}\sum_{\text{верно}} w_i + e^{\alpha}\sum_{\text{ошибки}} w_i = e^{-\alpha} W + (e^{\alpha} - e^{-\alpha})\sum_{\text{ошибки}} w_i,$$

где $W = \sum_i w_i$. При $\alpha > 0$ множитель $e^{\alpha} - e^{-\alpha}$ положителен, поэтому лучший $b$ — классификатор
с наименьшей **взвешенной долей ошибок** $\text{err} = \sum_{\text{ошибки}} w_i / W$. Подстановка в сумму
и деление на $W$: $f(\alpha) = e^{-\alpha}(1 - \text{err}) + e^{\alpha}\,\text{err}$. Производная
$f'(\alpha) = -e^{-\alpha}(1 - \text{err}) + e^{\alpha}\,\text{err} = 0$ при $e^{2\alpha} = \frac{1 - \text{err}}{\text{err}}$:

$$\alpha_t = \frac{1}{2}\ln\frac{1 - \text{err}_t}{\text{err}_t}.$$

**Вес классификатора на числах.** Классификатор, который угадывает как монетка,
$\text{err}_t = 0.5$, получает $\alpha_t = \frac{1}{2}\ln 1 = 0$. При трех ошибках из 14 и равных весах
$\text{err}_t = \frac{3}{14} = 0.214$ и $\alpha_t = \frac{1}{2}\ln\frac{0.786}{0.214} = \frac{1}{2}\ln 3.67 = 0.65$. При $\text{err}_t \to 0$
вес $\alpha_t \to \infty$.

**Пересчет весов.** $w_i^{(t+1)} = e^{-y_i a_t(x_i)} = w_i^{(t)}\, e^{-\alpha_t y_i b_t(x_i)}$: веса ошибок умножаются на $e^{\alpha_t}$,
верных — на $e^{-\alpha_t}$. Следующий классификатор обучают на взвешенной выборке,
в которой трудные объекты весят больше. Из формулы для $\alpha_t$

$$e^{\alpha_t} = \sqrt{\frac{1 - \text{err}_t}{\text{err}_t}}.$$

Сумма весов ошибок после умножения:

$$\text{err}_t \cdot \sqrt{\frac{1 - \text{err}_t}{\text{err}_t}} = \sqrt{\text{err}_t\,(1 - \text{err}_t)};$$

сумма весов верно классифицированных объектов:

$$(1 - \text{err}_t) \cdot \sqrt{\frac{\text{err}_t}{1 - \text{err}_t}} = \sqrt{\text{err}_t\,(1 - \text{err}_t)}.$$

Суммы равны: после пересчета на ошибки классификатора $b_t$ приходится ровно половина веса,
и с новыми весами он угадывает как монетка. Следующий классификатор вынужден ошибаться
на других объектах.

**Пример на плоскости.** 14 точек, класс 1 — выполнены хотя бы два условия из трех:
$x_1 < 6$, $x_2 < 6$, $x_1 > 2.5$. Один пень так не разделит. Веса перед каждым шагом считают
по формуле $w_i \propto e^{-y_i a_{t-1}(x_i)}$, доли ошибок сверяют с `AdaBoostClassifier`:

In [ ]:
rng_ada_toy = np.random.default_rng(61)
X_ada_toy = np.round(rng_ada_toy.uniform(0, 10, size=(14, 2)), 1)

# класс 1 — выполнены хотя бы два условия из трех
conditions_met = (X_ada_toy[:, 0] < 6).astype(int) + (X_ada_toy[:, 1] < 6).astype(int) + (X_ada_toy[:, 0] > 2.5).astype(int)
y_ada_toy = (conditions_met >= 2).astype(int)

ada_toy = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=3, learning_rate=1.0, random_state=SEED)
ada_toy.fit(X_ada_toy, y_ada_toy)

ada_toy_alphas = ada_toy.estimator_weights_ / 2                      # в sklearn вес пня ln((1 − err) / err) = 2α
ada_toy_signs = 2 * y_ada_toy - 1
ada_toy_stump_signs = np.array([2 * stump.predict(X_ada_toy) - 1 for stump in ada_toy.estimators_])     # b_t(x_i) ∈ {−1, +1}

# веса перед шагом t по формуле w_i ∝ exp(−y_i a_{t−1}(x_i)), a_0 = 0
ada_toy_weights = []
ada_toy_errors = []
ada_toy_scores = np.zeros(len(y_ada_toy))
for alpha, answers in zip(ada_toy_alphas, ada_toy_stump_signs):
    unnormalized = np.exp(-ada_toy_signs * ada_toy_scores)
    weights = unnormalized / unnormalized.sum()
    ada_toy_weights.append(weights)
    ada_toy_errors.append(weights[answers != ada_toy_signs].sum())
    ada_toy_scores = ada_toy_scores + alpha * answers

assert np.allclose(ada_toy_errors, ada_toy.estimator_errors_), "доли ошибок по формуле весов должны совпасть с sklearn"
print("доли ошибок по формуле весов совпадают с AdaBoostClassifier")

ada_toy_table = pd.DataFrame({
    "вопрос пня": [f"x{stump.tree_.feature[0] + 1} ≤ {stump.tree_.threshold[0]:.2f}" for stump in ada_toy.estimators_],
    "число ошибок пня": [int(np.sum(answers != ada_toy_signs)) for answers in ada_toy_stump_signs],
    "взвешенная доля ошибок err": ada_toy_errors,
    "вес пня α": ada_toy_alphas,
}, index=pd.RangeIndex(1, len(ada_toy_alphas) + 1, name="шаг t"))
show_table(ada_toy_table)

In [ ]:
def plot_adaboost_steps(X, y, stumps, alphas, errors, weights_by_step):
    """AdaBoost по шагам: размер точки — вес объекта перед шагом, кольцо — ошибка пня; справа — знак суммы."""
    point_colors = ListedColormap([BLUE, OCHRE])
    low, high = X.min() - 0.8, X.max() + 0.8
    grid_1, grid_2 = np.meshgrid(np.linspace(low, high, 300), np.linspace(low, high, 300))
    grid = np.column_stack([grid_1.ravel(), grid_2.ravel()])
    grid_scores = np.zeros(len(grid))
    train_scores = np.zeros(len(y))

    fig, axes = plt.subplots(1, len(stumps) + 1, figsize=(4.4 * (len(stumps) + 1), 4.9))
    for step, (ax, stump, alpha, error, weights) in enumerate(
            zip(axes, stumps, alphas, errors, weights_by_step), start=1):
        grid_classes = stump.predict(grid)
        grid_scores = grid_scores + alpha * (2 * grid_classes - 1)
        train_scores = train_scores + alpha * (2 * stump.predict(X) - 1)

        ax.contourf(grid_1, grid_2, grid_classes.reshape(grid_1.shape), levels=[-0.5, 0.5, 1.5],
                    colors=[BLUE, OCHRE], alpha=0.15)
        feature = stump.tree_.feature[0]
        threshold = stump.tree_.threshold[0]
        if feature == 0:
            ax.axvline(threshold, color=BLACK, lw=1.5)
        else:
            ax.axhline(threshold, color=BLACK, lw=1.5)

        sizes = 2600 * weights
        is_wrong = stump.predict(X) != y
        ax.scatter(X[:, 0], X[:, 1], s=sizes, c=y, cmap=point_colors, vmin=0, vmax=1, edgecolors="white",
                   lw=1, zorder=3)
        ax.scatter(X[is_wrong, 0], X[is_wrong, 1], s=2.2 * sizes[is_wrong] + 80, facecolors="none",
                   edgecolors=BLACK, lw=1.6, zorder=4)
        ax.set_title(f"шаг {step}: пень $x_{feature + 1} \\leq {threshold:.2f}$\n"
                     f"err = {error:.3f}, $\\alpha$ = {alpha:.2f}", fontsize=11)

    final = axes[-1]
    final.contourf(grid_1, grid_2, (grid_scores > 0).astype(int).reshape(grid_1.shape), levels=[-0.5, 0.5, 1.5],
                   colors=[BLUE, OCHRE], alpha=0.15)
    for stump in stumps:
        feature = stump.tree_.feature[0]
        threshold = stump.tree_.threshold[0]
        if feature == 0:
            final.axvline(threshold, color=GREY, lw=1, ls="--")
        else:
            final.axhline(threshold, color=GREY, lw=1, ls="--")
    final.scatter(X[:, 0], X[:, 1], s=90, c=y, cmap=point_colors, vmin=0, vmax=1, edgecolors="white", lw=1, zorder=3)
    mistakes = int(np.sum((train_scores > 0).astype(int) != y))
    terms = " + ".join(f"{alpha:.2f}\\,b_{step}" for step, alpha in enumerate(alphas, start=1))
    final.set_title(f"$a(x) = {terms}$\nошибок на обучении: {mistakes}", fontsize=11)

    for ax in axes:
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_aspect("equal")
        ax.set_xlabel("$x_1$")
        ax.grid(False)
    axes[0].set_ylabel("$x_2$")
    handles = [
        plt.Line2D([], [], marker="o", ls="", color=OCHRE, markersize=9, label="класс 1"),
        plt.Line2D([], [], marker="o", ls="", color=BLUE, markersize=9, label="класс 0"),
        plt.Line2D([], [], marker="o", ls="", markerfacecolor="none", markeredgecolor=BLACK, markersize=12,
                   label="ошибка пня этого шага"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=10, frameon=False)
    plt.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()


plot_adaboost_steps(X_ada_toy, y_ada_toy, ada_toy.estimators_, ada_toy_alphas, ada_toy_errors, ada_toy_weights)

Размер точки — вес объекта перед шагом, кольцо — ошибка пня этого шага. Первый пень
ошибается на трех точках из 14. На шаге 2 эти точки крупнее: на них приходится половина
веса, $\frac{1}{6}$ на каждую, и второй пень задает другой вопрос, на котором все три точки
классифицированы верно. Каждый пень ошибается на трех–шести точках, а знак суммы
$0.65\, b_1 + 0.75\, b_2 + 0.80\, b_3$ (справа) верно делит все 14 точек: области «хотя бы два
условия из трех» нет ни у одного пня, но она есть у их взвешенной суммы.

**Что оценивает AdaBoost.** Минимум ожидаемой потери $\mathbb{E}(e^{-y a} \mid x) = P(y = 1 \mid x)\,e^{-a} + P(y = -1 \mid x)\,e^{a}$
по $a$: производная $-P(y = 1 \mid x)e^{-a} + P(y = -1 \mid x)e^{a} = 0$ дает
$a^*(x) = \frac{1}{2}\ln\frac{P(y = 1 \mid x)}{P(y = -1 \mid x)}$ — половину логарифма отношения шансов; минимум ожидаемой
логистической потери — тот же логарифм без множителя $\frac{1}{2}$. Экспоненциальная потеря растет при ошибках
быстрее логистической, поэтому AdaBoost чувствительнее к выбросам и шуму в метках.

**Выбросы.** Вес объекта $e^{-M_i}$ растет экспоненциально с уменьшением **отступа** (margin)
$M_i = y_i\, a(x_i)$; отрицательный отступ — ошибка ансамбля. Объект с неверной меткой ансамбль
раз за разом классифицирует «неверно», его отступ уходит в минус, вес растет, и поздние
классификаторы подстраиваются под такие объекты. У логистической потери псевдоостаток
ограничен, $|y - p| \le 1$ (раздел 4), поэтому градиентный бустинг с ней устойчивее к ошибкам
разметки.

Реализация на данных Хасти и сравнение с `AdaBoostClassifier`; у sklearn
$\alpha_t = \ln\frac{1 - \text{err}_t}{\text{err}_t}$, вдвое больше, что не меняет знака суммы.

In [ ]:
def adaboost_stumps(X, y_signs, n_rounds):
    """AdaBoost на пнях; y_signs — метки −1 и +1. Возвращает пни, их веса α и веса объектов после каждого шага."""
    weights = np.full(len(y_signs), 1 / len(y_signs))
    stumps = []
    alphas = []
    weights_history = []

    for t in range(n_rounds):
        stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
        stump.fit(X, y_signs, sample_weight=weights)
        prediction = stump.predict(X)                          # −1 или +1

        is_wrong = prediction != y_signs
        error = weights[is_wrong].sum() / weights.sum()
        alpha = 0.5 * np.log((1 - error) / error)

        weights = weights * np.exp(-alpha * y_signs * prediction)
        weights = weights / weights.sum()

        stumps.append(stump)
        alphas.append(alpha)
        weights_history.append(weights)

    result = (stumps, np.array(alphas), weights_history)
    return result


def adaboost_score(stumps, alphas, X):
    score = np.zeros(len(X))
    for stump, alpha in zip(stumps, alphas):
        score = score + alpha * stump.predict(X)
    return score


yh_train_signs = 2 * yh_train - 1                            # 0 и 1 → −1 и +1
ada_stumps, ada_alphas, _ = adaboost_stumps(Xh_train, yh_train_signs, 400)

library_ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=400, learning_rate=1.0, random_state=SEED)
library_ada.fit(Xh_train, yh_train)

mine_scores = adaboost_score(ada_stumps, ada_alphas, Xh_test)
mine_labels = (mine_scores > 0).astype(int)
library_labels = library_ada.predict(Xh_test)
agreement = np.mean(mine_labels == library_labels)
alpha_ratio = library_ada.estimator_weights_[:5] / ada_alphas[:5]
ada_error = np.mean(mine_labels != yh_test)

print(f"совпадение предсказаний с AdaBoostClassifier на тестовом сете: {agreement:.4f}")
print("отношение α sklearn к своим на первых шагах:", alpha_ratio.round(3))
print(f"доля ошибок AdaBoost из 400 пней: {ada_error:.3f}")

Предсказания совпали с `AdaBoostClassifier` на всех 10 000 объектах тестового сета, веса
классификаторов у sklearn ровно вдвое больше. Доля ошибок AdaBoost из 400 пней — 0.116,
вдвое больше, чем у градиентного бустинга пней с логистической потерей и $\eta = 1$ (0.058)
на тех же данных.

Веса объектов на двух лунах после 1, 3 и 10 шагов: размер точки пропорционален весу.

In [ ]:
X_moons_ada, y_moons_ada = make_moons(n_samples=200, noise=0.3, random_state=0)
moons_signs = 2 * y_moons_ada - 1
_, _, moons_weights = adaboost_stumps(X_moons_ada, moons_signs, 10)

point_colors = ListedColormap([BLUE, OCHRE])
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, step in zip(axes, [1, 3, 10]):
    weights = moons_weights[step - 1]
    sizes = 2000 * weights
    ax.scatter(X_moons_ada[:, 0], X_moons_ada[:, 1], c=y_moons_ada, cmap=point_colors, s=sizes,
               edgecolor="white", lw=0.4, alpha=0.8)
    heaviest_share = np.sort(weights)[-20:].sum()
    ax.set_title(f"шаг {step}: доля веса 20 самых тяжелых объектов {heaviest_share:.2f}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
plt.tight_layout()
plt.show()

Вначале у всех 200 объектов вес 0.005, и 20 объектов несут 0.1 веса. После первого шага
20 самых тяжелых объектов несут 0.28 веса — это объекты, на которых ошибся первый пень;
после третьего и десятого шагов — 0.41 и 0.42. Тяжелые объекты лежат у границы лун
и внутри чужой луны; следующие пни обучаются с наибольшим весом на них.

## 7. Регуляризация

Бустинг, в отличие от леса, с ростом числа деревьев переобучается. Ограничивают его
четырьмя способами: темп обучения, глубина деревьев, число деревьев и случайные
подвыборки. Сравнение — на данных Фридмана из занятия 8:

$$y = 10\sin(\pi x_1 x_2) + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 1),$$

10 признаков, равномерных на $[0, 1]$, из них $x_6, \dots, x_{10}$ — шум; 500 объектов обучения
и 5000 тестового сета. Все модели записывают в общую таблицу `friedman_results`.

In [ ]:
Xf, yf = make_friedman1(n_samples=5500, noise=1.0, random_state=0)

Xf_train, yf_train = Xf[:500], yf[:500]           # 500 объектов обучения
Xf_test, yf_test = Xf[500:], yf[500:]             # 5000 объектов тестового сета

friedman_results = {}


def evaluate_regressor(name, model):
    """Обучить модель на данных Фридмана и записать в таблицу MSE на тестовом сете и время обучения."""
    started = time.perf_counter()
    model.fit(Xf_train, yf_train)
    fit_time = time.perf_counter() - started

    prediction = model.predict(Xf_test)
    test_mse = np.mean((prediction - yf_test) ** 2)

    friedman_results[name] = {"MSE на тесте": test_mse, "время обучения, с": fit_time}
    print(f"{name}: MSE на тесте {test_mse:.3f}, обучение {fit_time:.2f} с")
    return model


def staged_mse(model, X, y):
    """MSE после 1, 2, ..., T деревьев: staged_predict выдает предсказания по очереди."""
    errors = []
    for prediction in model.staged_predict(X):
        errors.append(np.mean((prediction - y) ** 2))

    errors = np.array(errors)
    return errors


print("обучение:", Xf_train.shape, " тест:", Xf_test.shape)

Для сравнения — линейная регрессия, одно дерево, случайный лес и бустинг с параметрами
sklearn по умолчанию: 100 деревьев глубины 3, $\eta = 0.1$.

In [ ]:
evaluate_regressor("линейная регрессия", LinearRegression())
evaluate_regressor("дерево, глубина 6", DecisionTreeRegressor(max_depth=6, random_state=SEED))
evaluate_regressor("случайный лес, 300 деревьев", RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1))
default_boosting = evaluate_regressor("бустинг, 100 деревьев глубины 3", GradientBoostingRegressor(random_state=SEED))

show_table(pd.DataFrame(friedman_results).T)

MSE на тестовом сете: 7.11 у линейной регрессии, 9.20 у дерева глубины 6, 4.45 у леса
и 2.81 у бустинга с параметрами по умолчанию; нижняя граница — дисперсия шума 1.

**Темп обучения** (shrinkage) $\eta$: $a_t = a_{t-1} + \eta\, b_t$. Фридман (2001) обнаружил опытным путем,
что малые $\eta$ дают меньшую ошибку на новых данных при большем числе деревьев.
Объяснение через связь с лассо: пошаговое построение с бесконечно малым шагом
для линейных базовых моделей проходит почти тот же путь решений, что лассо
при убывании штрафа (Hastie, Tibshirani, Friedman, ESL, раздел 16.2), то есть малый
шаг действует как регуляризация. Кривые до 2000 деревьев глубины 3:

In [ ]:
rates = [1.0, 0.3, 0.1, 0.03]
max_trees = 2000
rate_curves = {}

for rate in rates:
    model = GradientBoostingRegressor(n_estimators=max_trees, learning_rate=rate, max_depth=3, random_state=SEED)
    model.fit(Xf_train, yf_train)
    rate_curves[rate] = staged_mse(model, Xf_test, yf_test)

rate_rows = []
for rate, curve in rate_curves.items():
    best_index = int(np.argmin(curve))
    rate_rows.append({
        "темп η": str(rate),
        "лучшее число деревьев": best_index + 1,
        "MSE при лучшем числе": curve[best_index],
        f"MSE при {max_trees} деревьях": curve[-1],
    })
rate_table = pd.DataFrame(rate_rows).set_index("темп η")

trees_axis = np.arange(1, max_trees + 1)
plt.figure(figsize=(10, 4.8))
for (rate, curve), color in zip(rate_curves.items(), [OCHRE, GREY, BLUE, BLACK]):
    best_index = int(np.argmin(curve))
    plt.semilogx(trees_axis, curve, color=color, lw=1.8, label=f"η = {rate}")
    plt.scatter(best_index + 1, curve[best_index], color=color, s=40, zorder=3)
plt.axhline(1.0, color=GREY, ls=":", label=r"дисперсия шума $\sigma_\varepsilon^2 = 1$")
plt.ylim(0, 12)
plt.xlabel("число деревьев T")
plt.ylabel("MSE на тестовом сете")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

show_table(rate_table)

Минимумы MSE на тестовом сете: 6.63 при 9 деревьях для $\eta = 1$, 3.29 при 151 для $\eta = 0.3$,
2.60 при 438 для $\eta = 0.1$ и 2.53 при 836 для $\eta = 0.03$: чем меньше $\eta$, тем больше деревьев
до минимума и тем он ниже. Заметный рост после минимума — только при $\eta = 1$, с 6.63 до 7.41.

**Глубина и порядок взаимодействий.** Дерево глубины $d$ на любом пути задает не больше
$d$ вопросов и описывает взаимодействие не больше $d$ признаков. Пни дают аддитивную модель
$a(x) = a_0 + \sum_j g_j(x_j)$, где $g_j$ — сумма пней по признаку $j$. Кривые для глубин от 1 до 6:

In [ ]:
depths = [1, 2, 3, 4, 6]
depth_curves = {}

for depth in depths:
    model = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.1, max_depth=depth, random_state=SEED)
    model.fit(Xf_train, yf_train)
    depth_curves[depth] = staged_mse(model, Xf_test, yf_test)

depth_rows = []
for depth, curve in depth_curves.items():
    best_index = int(np.argmin(curve))
    depth_rows.append({"глубина": depth, "лучшее число деревьев": best_index + 1, "MSE при лучшем числе": curve[best_index]})
depth_table = pd.DataFrame(depth_rows).set_index("глубина")

trees_axis = np.arange(1, 1001)
plt.figure(figsize=(10, 4.8))
for (depth, curve), color in zip(depth_curves.items(), [OCHRE, GREY, BLUE, BLACK, "#4DAF4A"]):
    plt.semilogx(trees_axis, curve, color=color, lw=1.8, label=f"глубина {depth}")
plt.axhline(1.0, color=GREY, ls=":", label=r"дисперсия шума $\sigma_\varepsilon^2 = 1$")
plt.ylim(0, 12)
plt.xlabel("число деревьев T, темп η = 0.1")
plt.ylabel("MSE на тестовом сете")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

show_table(depth_table)

Пни не описывают $\sin(\pi x_1 x_2)$ и ошибаются больше (3.46); глубина 2–3 дает 2.60–2.65,
глубина 6 — 3.56. Фридман рекомендовал деревья с 4–8 листьями: взаимодействия высокого
порядка в реальных данных редки, а глубокие деревья подгоняются под шум.

**Ранняя остановка.** Часть обучающей выборки откладывают как валидационную
и после каждого дерева вычисляют потерю на ней. В sklearn обучение прекращают, когда
потеря не меньше (с запасом `tol`) ни одного из последних `n_iter_no_change` значений;
лучшая итерация не запоминается, и в модели остаются все построенные деревья.
LightGBM (`early_stopping`) и CatBoost (`use_best_model`) запоминают лучшую итерацию
и предсказывают по ней.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.6), sharey=True)
stopping_rows = []

for ax, rate in zip(axes, [0.1, 1.0]):
    model = GradientBoostingRegressor(n_estimators=2000, learning_rate=rate, max_depth=3,
                                      validation_fraction=0.2, n_iter_no_change=20, random_state=SEED)
    model.fit(Xf_train, yf_train)

    stopped_trees = model.n_estimators_
    stopped_prediction = model.predict(Xf_test)
    stopped_mse = np.mean((stopped_prediction - yf_test) ** 2)
    curve = rate_curves[rate]
    best_index = int(np.argmin(curve))

    stopping_rows.append({
        "темп η": str(rate),
        "деревьев после остановки": stopped_trees,
        "MSE после остановки": stopped_mse,
        "лучшее число деревьев по тесту": best_index + 1,
        "MSE при лучшем числе": curve[best_index],
    })

    trees_axis = np.arange(1, len(curve) + 1)
    ax.semilogx(trees_axis, curve, color=BLUE, lw=1.8, label="MSE на тестовом сете, все 2000 деревьев")
    ax.axvline(stopped_trees, color=OCHRE, lw=2, label=f"ранняя остановка: {stopped_trees} деревьев")
    ax.scatter(best_index + 1, curve[best_index], color=BLACK, s=40, zorder=3, label=f"минимум: {best_index + 1} деревьев")
    ax.set_title(f"η = {rate}", fontsize=12)
    ax.set_xlabel("число деревьев T")
    ax.legend()

axes[0].set_ylabel("MSE на тестовом сете")
axes[0].set_ylim(0, 12)
plt.tight_layout()
plt.show()

stopping_table = pd.DataFrame(stopping_rows).set_index("темп η")
show_table(stopping_table)

При $\eta = 0.1$ остановка на 305 деревьях с MSE 2.77; модель обучена на 400 объектах
из 500, и ошибка выше минимума кривой модели на всех объектах (2.60). При $\eta = 1$
потеря на валидации наименьшая после 5 деревьев (MSE на тестовом сете 8.48), но колеблется,
и остановка наступает только на 82 деревьях: после лучшей итерации в модели остаются
77 деревьев, и MSE равна 10.51.

**Стохастический градиентный бустинг** (Friedman, 2002): каждое дерево обучают на случайной
доле `subsample` объектов без возвращения. Это уменьшает корреляцию деревьев, как
в бэггинге, ускоряет обучение и дает оценку улучшения на объектах вне подвыборки
(`oob_improvement_`).

In [ ]:
subsample_curves = {}

for share in [1.0, 0.5, 0.2]:
    model = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.1, max_depth=3, subsample=share, random_state=SEED)
    model.fit(Xf_train, yf_train)
    subsample_curves[share] = staged_mse(model, Xf_test, yf_test)

trees_axis = np.arange(1, 1001)
plt.figure(figsize=(10, 4.5))
for (share, curve), color in zip(subsample_curves.items(), [BLACK, BLUE, OCHRE]):
    best_index = int(np.argmin(curve))
    plt.semilogx(trees_axis, curve, color=color, lw=1.8, label=f"subsample = {share}: минимум {curve[best_index]:.3f}")
plt.xlim(10, 1000)
plt.ylim(1.5, 6)
plt.xlabel("число деревьев T, темп η = 0.1")
plt.ylabel("MSE на тестовом сете")
plt.legend()
plt.show()

Половина выборки на дерево снижает наименьшую MSE с 2.60 до 2.47; доля 0.2 повышает ее
до 2.63: каждое дерево обучено на 100 объектах. Кроме подвыборки объектов, используют
подвыборку признаков: на дерево (`colsample_bytree` в XGBoost, `feature_fraction` в LightGBM)
или в каждом узле, как в лесу (`max_features` в sklearn, `colsample_bynode` в XGBoost,
`feature_fraction_bynode` в LightGBM).

In [ ]:
evaluate_regressor(
    "бустинг, ранняя остановка",
    GradientBoostingRegressor(n_estimators=2000, validation_fraction=0.2, n_iter_no_change=20, random_state=SEED),
)
evaluate_regressor(
    "стохастический бустинг, subsample = 0.5, ранняя остановка",
    GradientBoostingRegressor(n_estimators=2000, subsample=0.5, validation_fraction=0.2, n_iter_no_change=20,
                              random_state=SEED),
)
show_table(pd.DataFrame(friedman_results).T)

С ранней остановкой MSE 2.77, со стохастическим бустингом 2.79 — почти как у бустинга
с параметрами по умолчанию (2.81), но число деревьев выбрано по валидационной выборке.

**Нужны ли объекты, на которых модель уже не ошибается?** В градиентном бустинге
каждое дерево обучают на всех объектах обучающей выборки, в том числе на тех,
которые модель уже предсказывает почти точно: их псевдоостатки близки к нулю.
Опыт на данных Фридмана: перед обучением каждого дерева отбрасывают долю объектов
с наименьшими по модулю остатками, дерево обучают на остальных, а предсказание
обновляют на всех объектах; 400 деревьев глубины 3, $\eta = 0.1$.

In [ ]:
def boosting_without_small_residuals(drop_share, n_trees=400, learning_rate=0.1):
    """Бустинг на данных Фридмана, в котором каждое дерево обучают без доли drop_share объектов
    с наименьшими по модулю остатками; предсказание обновляют на всех объектах."""
    current_train = np.full(len(yf_train), yf_train.mean())
    current_test = np.full(len(yf_test), yf_train.mean())

    for t in range(n_trees):
        residuals = yf_train - current_train
        size = np.abs(residuals)
        cutoff = np.quantile(size, drop_share)
        keep = size >= cutoff

        tree = DecisionTreeRegressor(max_depth=3, random_state=SEED)
        tree.fit(Xf_train[keep], residuals[keep])

        current_train = current_train + learning_rate * tree.predict(Xf_train)
        current_test = current_test + learning_rate * tree.predict(Xf_test)

    test_mse = np.mean((current_test - yf_test) ** 2)
    return test_mse


drop_shares = [0.0, 0.3, 0.5, 0.8, 0.95]
drop_results = {}
for share in drop_shares:
    drop_results[share] = boosting_without_small_residuals(share)

plt.figure(figsize=(9, 4))
labels = [f"{int(share * 100)}%" for share in drop_shares]
plt.bar(labels, list(drop_results.values()), color=[BLACK] + [OCHRE] * (len(drop_shares) - 1))
plt.axhline(drop_results[0.0], color=BLACK, ls=":", lw=1)
plt.xlabel("доля объектов с наименьшими |остатками|, которых нет в обучении каждого дерева")
plt.ylabel("MSE на тестовом сете")
plt.show()

drop_table = pd.DataFrame({"MSE на тесте": list(drop_results.values())}, index=labels)
drop_table.index.name = "доля отброшенных"
show_table(drop_table)

Если отбросить 30–50% объектов с наименьшими остатками, MSE почти не меняется (2.60 и 2.58
против 2.62): остатки этих объектов близки к нулю, и без них дерево почти такое же. На этом
построен GOSS (gradient-based one-side sampling, односторонняя выборка по градиенту)
в LightGBM: часть объектов с малыми градиентами отбрасывают, а оставшимся из них
увеличивают вес, чтобы суммы градиентов в листьях не сместились. Если отбросить 80% и 95%,
MSE растет до 3.38 и 7.42: значение листа — среднее остатков только трудных объектов,
и модель сдвигается слишком сильно на всех объектах листа, в том числе там, где она уже
была права. Поэтому в обычном градиентном бустинге каждое дерево обучают на всех объектах:
объекты без ошибок удерживают модель там, где она уже точна.

## 8. Смещение и разброс бустинга

Разложение ошибки (занятие 8): шум $\sigma_\varepsilon^2$, смещение² и разброс по 300 обучающим выборкам
синуса из 80 точек. Пень, бэггинг из 100 пней и бустинг пней с $\eta = 0.1$ после 1–300
деревьев на тех же выборках:

In [ ]:
def simulate_bias_variance(make_model, n_datasets=300, n_train=80, noise=0.25, seed=0):
    """Смещение², разброс и ошибка модели на синусе по многим обучающим выборкам."""
    rng = np.random.default_rng(seed)

    x_eval = np.linspace(0, 2 * np.pi, 200)[:, None]      # (200, 1): точки, где оценивают слагаемые
    f_eval = np.sin(x_eval[:, 0])                         # (200,): истинная функция в этих точках

    predictions = np.empty((n_datasets, len(x_eval)))     # строка b — предсказания модели, обученной на выборке b
    for b in range(n_datasets):
        x_sample = rng.uniform(0, 2 * np.pi, n_train)[:, None]
        noise_sample = rng.normal(0, noise, n_train)
        y_sample = np.sin(x_sample[:, 0]) + noise_sample

        model = make_model()
        model.fit(x_sample, y_sample)
        predictions[b] = model.predict(x_eval)

    mean_prediction = predictions.mean(axis=0)            # (200,): среднее предсказание по выборкам
    bias_squared = np.mean((mean_prediction - f_eval) ** 2)
    variance = np.mean(predictions.var(axis=0))

    y_new = f_eval + rng.normal(0, noise, predictions.shape)
    direct_error = np.mean((y_new - predictions) ** 2)

    estimate = {
        "смещение²": bias_squared,
        "разброс": variance,
        "дисперсия шума": noise ** 2,
        "сумма трех слагаемых": bias_squared + variance + noise ** 2,
        "ошибка напрямую": direct_error,
    }
    result = (estimate, x_eval, predictions)
    return result


def simulate_boosting_stages(stages, learning_rate=0.1, n_datasets=300, n_train=80, noise=0.25, seed=0):
    """Смещение², разброс и ошибка бустинга пней после stages[k] деревьев; выборки те же, что в simulate_bias_variance."""
    rng = np.random.default_rng(seed)

    x_eval = np.linspace(0, 2 * np.pi, 200)[:, None]
    f_eval = np.sin(x_eval[:, 0])

    predictions = np.empty((len(stages), n_datasets, len(x_eval)))     # [число деревьев, выборка, точка]
    for b in range(n_datasets):
        x_sample = rng.uniform(0, 2 * np.pi, n_train)[:, None]
        noise_sample = rng.normal(0, noise, n_train)
        y_sample = np.sin(x_sample[:, 0]) + noise_sample

        model = GradientBoostingRegressor(n_estimators=max(stages), learning_rate=learning_rate, max_depth=1, random_state=SEED)
        model.fit(x_sample, y_sample)

        position = 0
        for number, stage_prediction in enumerate(model.staged_predict(x_eval), start=1):
            if number == stages[position]:
                predictions[position, b] = stage_prediction
                position = position + 1
                if position == len(stages):
                    break

    y_new = f_eval + rng.normal(0, noise, (n_datasets, len(x_eval)))
    rows = []
    for position, n_trees in enumerate(stages):
        stage_predictions = predictions[position]
        mean_prediction = stage_predictions.mean(axis=0)

        bias_squared = np.mean((mean_prediction - f_eval) ** 2)
        variance = np.mean(stage_predictions.var(axis=0))
        direct_error = np.mean((y_new - stage_predictions) ** 2)

        rows.append({
            "модель": f"бустинг пней, T = {n_trees}",
            "смещение²": bias_squared,
            "разброс": variance,
            "дисперсия шума": noise ** 2,
            "сумма трех слагаемых": bias_squared + variance + noise ** 2,
            "ошибка напрямую": direct_error,
        })

    table = pd.DataFrame(rows).set_index("модель")
    return table


def make_stump():
    model = DecisionTreeRegressor(max_depth=1, random_state=SEED)
    return model


def make_bagged_stumps():
    model = BaggingRegressor(DecisionTreeRegressor(max_depth=1), n_estimators=100, random_state=SEED)
    return model


stump_estimate, _, _ = simulate_bias_variance(make_stump)
bagging_estimate, _, _ = simulate_bias_variance(make_bagged_stumps)

boosting_stages = [1, 3, 10, 30, 100, 200, 300]
boosting_table = simulate_boosting_stages(boosting_stages)

reference_rows = pd.DataFrame([stump_estimate, bagging_estimate], index=["пень", "бэггинг, 100 пней"])
boosting_bias_variance = pd.concat([reference_rows, boosting_table])
show_table(boosting_bias_variance, digits=4)

In [ ]:
plt.figure(figsize=(9, 4.4))
for column, color in [("смещение²", BLUE), ("разброс", OCHRE), ("сумма трех слагаемых", BLACK)]:
    plt.semilogx(boosting_stages, boosting_table[column], "o-", color=color, label=f"бустинг: {column}")
    plt.axhline(bagging_estimate[column], color=color, ls=":", lw=1.2)
plt.axhline(0.25 ** 2, color=GREY, ls="--", label=r"дисперсия шума $\sigma_\varepsilon^2$")
plt.plot([], [], color=GREY, ls=":", label="бэггинг 100 пней (те же цвета)")
plt.xticks(boosting_stages, boosting_stages)
plt.minorticks_off()
plt.xlabel("число пней T, темп η = 0.1")
plt.ylabel("слагаемые ошибки, среднее по x")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

Бэггинг пней уменьшает разброс с 0.031 до 0.015 и почти не меняет смещение² (0.075 и 0.073):
среднее одинаково смещенных моделей смещено так же, $\mathbb{E}\,a = \mathbb{E}\,b_1$. Бустинг уменьшает смещение²
с 0.420 до 0.0018 при 200 пнях, а разброс растет с 0.007 до 0.016: каждое дерево
подгоняется под остатки своей выборки. Наименьшая сумма — 0.080 при 200 пнях, меньше
лучшего одного дерева (0.096) и бэггинга глубоких деревьев (0.095) из занятия 8. Отсюда
правило выбора базовой модели: для бэггинга и леса — глубокие деревья с малым
смещением, для бустинга — неглубокие с малым разбросом.

## 9. Второй порядок и регуляризация листьев: XGBoost

XGBoost (extreme gradient boosting; Chen, Guestrin, 2016) на каждом шаге минимизирует
потерю вместе со штрафом за сложность дерева $f$ с $J$ листьями и значениями $w_j$:

$$\mathcal{L}_t = \sum_{i=1}^{n} L\big(y_i,\ a_{t-1}(x_i) + f(x_i)\big) + \mu J + \frac{\lambda}{2}\sum_{j=1}^{J} w_j^2.$$

**Разложение второго порядка.** С $g_i = \frac{\partial L}{\partial a}$ и $h_i = \frac{\partial^2 L}{\partial a^2}$ в точке $a_{t-1}(x_i)$:

$$\mathcal{L}_t \approx \sum_i \Big(L(y_i, a_{t-1}(x_i)) + g_i f(x_i) + \tfrac{1}{2}h_i f(x_i)^2\Big) + \mu J + \frac{\lambda}{2}\sum_j w_j^2.$$

Первое слагаемое от $f$ не зависит. Дерево принимает значение $w_j$ на объектах листа $R_j$,
поэтому суммы по объектам группируются по листьям, с $G_j = \sum_{x_i \in R_j} g_i$ и $H_j = \sum_{x_i \in R_j} h_i$:

$$\tilde{\mathcal{L}}_t = \sum_{j=1}^{J}\Big(G_j w_j + \tfrac{1}{2}(H_j + \lambda)\,w_j^2\Big) + \mu J.$$

**Значение листа.** Каждое слагаемое — квадратичная функция своего $w_j$; производная
$G_j + (H_j + \lambda)w_j = 0$ дает

$$w_j^* = -\frac{G_j}{H_j + \lambda}.$$

**Качество структуры.** Подстановка $w_j^*$: $G_j w_j^* + \frac{1}{2}(H_j + \lambda)(w_j^*)^2 = -\frac{G_j^2}{H_j + \lambda} + \frac{1}{2}\frac{G_j^2}{H_j + \lambda} = -\frac{1}{2}\frac{G_j^2}{H_j + \lambda}$,

$$\tilde{\mathcal{L}}_t^* = -\frac{1}{2}\sum_{j=1}^{J}\frac{G_j^2}{H_j + \lambda} + \mu J.$$

**Прирост разбиения.** Лист с суммами $G = G_l + G_r$, $H = H_l + H_r$ делят на два. Слагаемое листа
$-\frac{1}{2}\frac{G^2}{H + \lambda} + \mu$ заменяется двумя, $-\frac{1}{2}\frac{G_l^2}{H_l + \lambda} - \frac{1}{2}\frac{G_r^2}{H_r + \lambda} + 2\mu$; уменьшение $\tilde{\mathcal{L}}_t^*$:

$$\text{Gain} = \frac{1}{2}\Big(\frac{G_l^2}{H_l + \lambda} + \frac{G_r^2}{H_r + \lambda} - \frac{G^2}{H + \lambda}\Big) - \mu.$$

Дерево растят жадно, как CART, но вместо уменьшения MSE используют Gain; разбиение
с $\text{Gain} < 0$ не делают, поэтому $\mu$ (`gamma` в XGBoost, `min_split_gain` в LightGBM) работает
как отсечение. Параметр `min_child_weight` — наименьшая допустимая $H$ листа.

**Частные случаи.** Для $L = \frac{1}{2}(y - a)^2$: $g_i = a_i - y_i$, $h_i = 1$, $H_j = |R_j|$, и $w_j^* = \frac{\sum_{x_i \in R_j}(y_i - a_i)}{|R_j| + \lambda}$ —
среднее остатков, стянутое к нулю тем сильнее, чем меньше объектов в листе; при $\lambda = 0$
Gain с точностью до множителя совпадает с уменьшением суммы квадратов ошибок.
`min_child_weight` здесь — наименьшее число объектов листа. Для логистической потери
$g_i = p_i - y_i$, $h_i = p_i(1 - p_i)$, и при $\lambda = 0$ значение листа — шаг Ньютона из раздела 4.

LightGBM использует те же формулы. Проверка: пень LightGBM (`num_leaves=2`, один шаг,
$\eta = 1$, начальная оценка 0) с $\lambda = 3$ на данных Хасти; значения его листьев сравнивают
с $-G/(H + \lambda)$, где при $a = 0$ $g_i = 0.5 - y_i$ и $h_i = 0.25$.

In [ ]:
leaf_lambda = 3.0
lgbm_stump = LGBMClassifier(n_estimators=1, learning_rate=1.0, num_leaves=2, reg_lambda=leaf_lambda,
                            min_child_samples=1, min_child_weight=0, boost_from_average=False, verbose=-1)
lgbm_stump.fit(Xh_train, yh_train)

leaf_index = lgbm_stump.predict(Xh_train, pred_leaf=True).ravel()       # номер листа каждого объекта
raw_scores = lgbm_stump.predict(Xh_train, raw_score=True)               # значение листа объекта

gradients = 0.5 - yh_train                                              # g_i = p_i − y_i при p_i = σ(0) = 0.5
hessians = np.full(len(yh_train), 0.25)                                 # h_i = p_i (1 − p_i)

xgb_rows = []
for leaf in np.unique(leaf_index):
    in_leaf = leaf_index == leaf
    gradient_sum = gradients[in_leaf].sum()
    hessian_sum = hessians[in_leaf].sum()

    xgb_rows.append({
        "лист": leaf,
        "объектов": in_leaf.sum(),
        "сумма g": gradient_sum,
        "сумма h": hessian_sum,
        "−G / (H + λ)": -gradient_sum / (hessian_sum + leaf_lambda),
        "значение листа в LightGBM": raw_scores[in_leaf][0],
    })

pd.DataFrame(xgb_rows).set_index("лист").round(5)

Значения совпадают. По сравнению с градиентным бустингом Фридмана XGBoost: использует вторую производную
и при выборе разбиения, а не только при пересчете листьев; регуляризует значения
листьев ($\lambda$, также L1-штраф `alpha`) и число листьев ($\mu$); ищет пороги по приближенным
квантилям признаков и умеет обрабатывать пропуски, выбирая для них направление
в каждом узле по Gain.

## 10. Гистограммы, LightGBM и CatBoost

**Гистограммы.** Точный перебор порогов требует сортировки объектов узла по каждому
признаку, $O(n \log n)$ на признак. Гистограммный бустинг до обучения один раз разбивает
каждый признак по квантилям на $B \le 255$ интервалов (бинов) и хранит вместо значения номер
бина. В узле за один проход по $n$ объектам для каждого бина накапливают $G$ и $H$, затем
перебирают $B - 1$ границу: $O(n + B)$ на признак. Гистограмму одного из потомков получают
вычитанием гистограммы другого из гистограммы родителя, поэтому считают только
меньшего потомка. Порог выбирают из 255 вариантов вместо всех значений признака;
на данных Фридмана ниже это почти не меняет качество. В sklearn — `HistGradientBoostingRegressor` и `HistGradientBoostingClassifier`.

**LightGBM** (Ke и соавторы, 2017):

- рост по листьям (leaf-wise): каждый раз делят лист с наибольшим Gain по всему
  дереву, а не все листья уровня; при том же числе листьев дерево сильнее уменьшает
  потерю на обучающей выборке и легче переобучается; размер задает `num_leaves`, глубину дополнительно ограничивают `max_depth`;
- GOSS (gradient-based one-side sampling, односторонняя выборка по градиенту):
  оставляют $a\,n$ объектов с наибольшими $|g_i|$, из остальных случайно берут $b\,n$ объектов
  ($a$ и $b$ — доли от всей выборки) и умножают их $g_i$ и $h_i$ на $\frac{1 - a}{b}$, чтобы суммы $G$ и $H$
  оставались несмещенными;
- EFB (exclusive feature bundling, объединение взаимоисключающих признаков):
  разреженные признаки, которые почти не бывают ненулевыми одновременно (например,
  one-hot столбцы), объединяют в один со сдвинутыми диапазонами бинов.

**CatBoost** (Прохоренкова и соавторы, 2018):

- симметричные деревья (oblivious trees): на каждом уровне один вопрос для всех узлов,
  дерево глубины $d$ — таблица из $2^d$ значений, номер листа — $d$ бит ответов. Такие деревья
  хуже подгоняются под отдельные объекты и быстро применяются;
- **смещение предсказаний** (prediction shift): псевдоостатки на шаге $t$ вычисляют
  по модели, обученной на тех же объектах, поэтому на обучающей выборке они меньше,
  чем на новых данных. Упорядоченный бустинг (ordered boosting, `boosting_type="Ordered"`)
  вычисляет псевдоостаток объекта моделью, обученной только на объектах, стоящих
  раньше него в случайной перестановке; в примерах ниже CatBoost работает в обычном
  режиме `Plain`, а упорядоченные статистики категорий применяются в обоих режимах;
- **упорядоченные целевые статистики** (ordered target statistics) для категориальных
  признаков: значение категории $c$ у $i$-го объекта перестановки заменяют на
  $\hat{x}_i = \frac{\sum_{j < i,\ x_j = c} y_j + \beta\, P}{\#\{j < i : x_j = c\} + \beta}$, где $P$ — среднее ответов, $\beta$ — вес априорного значения.
  Ответ самого объекта в его код не входит.

**Утечка при кодировании категорий.** Если заменить категорию средним ответом
по всей обучающей выборке, в код объекта входит его собственный ответ. У редких
категорий код почти равен ответу, модель опирается на него, а на тестовом сете
этой связи нет (занятие 4). Пример: ответ зависит от числового признака и от города
(20 значений); номер пользователя (1000 значений, по 2–3 объекта на значение) на ответ
не влияет.

In [ ]:
rng_categories = np.random.default_rng(3)
n_objects = 3000

city = rng_categories.integers(0, 20, n_objects)                  # полезная категория, 20 значений
user = rng_categories.integers(0, 1000, n_objects)                # шумовая категория, 1000 значений
numeric_feature = rng_categories.normal(size=n_objects)
city_effect = rng_categories.normal(0, 1, 20)
noise_categories = rng_categories.normal(0, 1, n_objects)
y_categories = 2 * numeric_feature + city_effect[city] + noise_categories

categories_frame = pd.DataFrame({"numeric": numeric_feature, "city": city, "user": user})
train_part = np.arange(n_objects) < 2000
frame_train, frame_test = categories_frame[train_part], categories_frame[~train_part]
y_cat_train, y_cat_test = y_categories[train_part], y_categories[~train_part]

print("объектов на одного пользователя в обучении:", round(2000 / frame_train["user"].nunique(), 2))

In [ ]:
def naive_target_encoding(train, test, y_train, columns):
    """Заменить категорию средним ответом по всей обучающей выборке, включая сам объект."""
    train_encoded = train.copy()
    test_encoded = test.copy()
    overall_mean = y_train.mean()

    for column in columns:
        means = pd.Series(y_train, index=train.index).groupby(train[column]).mean()
        train_encoded[column] = train[column].map(means)
        test_encoded[column] = test[column].map(means).fillna(overall_mean)

    encoded = (train_encoded, test_encoded)
    return encoded


def test_mse_of(model, train, test):
    model.fit(train, y_cat_train)
    prediction = model.predict(test)
    error = np.mean((prediction - y_cat_test) ** 2)
    return error


encoding_results = {}

naive_train, naive_test = naive_target_encoding(frame_train, frame_test, y_cat_train, ["city", "user"])
encoding_results["среднее по всей выборке"] = test_mse_of(
    HistGradientBoostingRegressor(max_iter=300, random_state=SEED), naive_train, naive_test)

cross_fitted = TargetEncoder(target_type="continuous", cv=KFold(5, shuffle=True, random_state=SEED))
cross_train = frame_train.copy()
cross_test = frame_test.copy()
cross_train[["city", "user"]] = cross_fitted.fit_transform(frame_train[["city", "user"]], y_cat_train)
cross_test[["city", "user"]] = cross_fitted.transform(frame_test[["city", "user"]])
encoding_results["TargetEncoder с кросс-фиттингом"] = test_mse_of(
    HistGradientBoostingRegressor(max_iter=300, random_state=SEED), cross_train, cross_test)

encoding_results["CatBoost, cat_features"] = test_mse_of(
    CatBoostRegressor(iterations=300, cat_features=["city", "user"], random_seed=SEED, verbose=0, allow_writing_files=False),
    frame_train, frame_test)

encoding_results["без пользователя, кросс-фиттинг города"] = test_mse_of(
    HistGradientBoostingRegressor(max_iter=300, random_state=SEED), cross_train.drop(columns="user"), cross_test.drop(columns="user"))

encoding_table = pd.DataFrame({"MSE на тесте": encoding_results})
show_table(encoding_table)

Дисперсия шума равна 1. Кодирование средним по всей выборке дает MSE 1.56: у пользователя
2–3 объекта, его код почти повторяет их ответы, и модель опирается на шумовой признак.
`TargetEncoder` с кросс-фиттингом вычисляет код объекта по остальным блокам выборки, без его
собственного ответа, и дает 1.29; модель без признака пользователя — 1.23. CatBoost
с упорядоченными статистиками — 1.13; это лучше модели без пользователя, но сравниваются
разные алгоритмы, а не только способы кодирования.

Время и качество четырех реализаций на 10 000 объектах данных Фридмана, одинаковые
300 деревьев глубины 3 и $\eta = 0.1$; сначала те же модели на 500 объектах в общей таблице.

In [ ]:
def make_library_models(n_trees=300):
    """Четыре реализации градиентного бустинга с одинаковыми настройками: n_trees деревьев глубины 3, η = 0.1."""
    models = {
        "GradientBoostingRegressor": GradientBoostingRegressor(
            n_estimators=n_trees, learning_rate=0.1, max_depth=3, random_state=SEED),
        "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
            max_iter=n_trees, learning_rate=0.1, max_depth=3, early_stopping=False, random_state=SEED),
        "LightGBM": LGBMRegressor(
            n_estimators=n_trees, learning_rate=0.1, max_depth=3, num_leaves=8, random_state=SEED, verbose=-1),
        "CatBoost": CatBoostRegressor(
            iterations=n_trees, learning_rate=0.1, depth=3, random_seed=SEED, verbose=0, allow_writing_files=False),
    }
    return models


for name, model in make_library_models().items():
    evaluate_regressor(name, model)

Те же четыре модели на 10 000 объектах обучения:

In [ ]:
X_big, y_big = make_friedman1(n_samples=15000, noise=1.0, random_state=1)
X_big_train, y_big_train = X_big[:10000], y_big[:10000]
X_big_test, y_big_test = X_big[10000:], y_big[10000:]

timing_rows = {}
for name, model in make_library_models().items():
    started = time.perf_counter()
    model.fit(X_big_train, y_big_train)
    fit_time = time.perf_counter() - started

    prediction = model.predict(X_big_test)
    test_mse = np.mean((prediction - y_big_test) ** 2)
    timing_rows[name] = {"MSE на тесте": test_mse, "время обучения, с": fit_time}

timing_table = pd.DataFrame(timing_rows).T
show_table(timing_table)

На 500 объектах MSE: 2.62 у `GradientBoostingRegressor`, 2.36 у гистограммной версии sklearn,
2.30 у LightGBM, 1.85 у CatBoost — регуляризация по умолчанию (не меньше 20 объектов
в листе у гистограммных реализаций, $\lambda = 3$ и симметричные деревья у CatBoost) помогает
на малой выборке. На 10 000 объектах качество почти одинаковое, а время обучения
различается в десятки раз.

## 11. Важность признаков

`feature_importances_` бустинга — MDI (mean decrease in impurity, среднее уменьшение
неоднородности; занятие 8), усредненная по деревьям: для каждого
признака суммируют уменьшение MSE по псевдоостаткам во всех узлах, где он задает
вопрос. В LightGBM `importance_type="split"` считает число разбиений по признаку,
`"gain"` — суммарный прирост. На данных Фридмана известна истина: $x_1, \dots, x_5$ влияют
на ответ, $x_6, \dots, x_{10}$ — нет. MDI и перестановочная важность (падение $R^2$ на тестовом
сете при перемешивании признака) бустинга из 400 деревьев:

In [ ]:
importance_model = GradientBoostingRegressor(n_estimators=400, learning_rate=0.1, max_depth=3, random_state=SEED)
importance_model.fit(Xf_train, yf_train)

permutation = permutation_importance(importance_model, Xf_test[:2000], yf_test[:2000], n_repeats=5, random_state=SEED)

feature_labels = [f"x{j + 1}" for j in range(10)]
importance_table = pd.DataFrame({
    "MDI": importance_model.feature_importances_,
    "перестановочная": permutation.importances_mean,
}, index=feature_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, column, color in zip(axes, importance_table.columns, [BLUE, OCHRE]):
    ax.bar(feature_labels, importance_table[column], color=color)
    ax.set_title(column, fontsize=12)
axes[0].set_ylabel("важность")
plt.tight_layout()
plt.show()

importance_table.round(3).T

Обе важности отделяют шумовые признаки: у $x_6, \dots, x_{10}$ MDI не больше 0.005, перестановочная
важность не больше 0.001. Порядок полезных признаков следует дисперсиям слагаемых формулы:
у $10\sin(\pi x_1 x_2)$ она 11.2 и делится между $x_1$ и $x_2$ поровну (MDI 0.221 и 0.218), у $10x_4$ — 8.3
(MDI 0.354, наибольшая у одного признака), у $20(x_3 - 0.5)^2$ и $5x_5$ — 2.2 и 2.1 (MDI 0.095 и 0.092).
Перестановочная важность дает тот же порядок групп: $x_4$, затем $x_1$ и $x_2$, затем $x_3$ и $x_5$;
внутри пар признаки различаются меньше чем на 0.02 и меняются местами.

## 12. Итоги

Итоговая таблица данных Фридмана:

In [ ]:
final_table = pd.DataFrame(friedman_results).T.sort_values("MSE на тесте")
show_table(final_table)

- бустинг — пошаговое построение суммы базовых моделей: каждое следующее дерево
  уменьшает потерю суммы предыдущих;
- градиентный бустинг — градиентный спуск в пространстве предсказаний: дерево
  приближает вектор антиградиента по MSE, значения листьев подбирают по исходной
  потере;
- псевдоостатки: $y - a$ для MSE, $\operatorname{sign}(y - a)$ для MAE, обрезанный остаток для функции
  Хьюбера, $\alpha$ или $\alpha - 1$ для квантильной потери, $y - p$ для логистической,
  $[y = k] - p_k$ для многоклассовой;
- начальное приближение — лучшая константа: среднее, медиана, квантиль, логарифм
  отношения шансов;
- значение листа классификатора — шаг Ньютона $-G/H$; XGBoost добавляет $\lambda$: $-G/(H + \lambda)$;
  прирост разбиения по второй производной — формула Gain со штрафом $\mu$;
- AdaBoost — пошаговое построение с экспоненциальной потерей: веса объектов
  $e^{-y a}$, вес классификатора $\frac{1}{2}\ln\frac{1 - \text{err}}{\text{err}}$;
- регуляризация: малый темп обучения с большим числом деревьев, неглубокие деревья,
  ранняя остановка по валидации, подвыборки объектов и признаков;
- бустинг уменьшает смещение, бэггинг — разброс;
- гистограммы, рост по листьям, GOSS и EFB ускоряют LightGBM; симметричные деревья,
  упорядоченный бустинг и упорядоченные целевые статистики — особенности CatBoost.